# Downloads

In [ ]:
!pip install -q diffusers>=0.32.2 transformers>=4.49.0 accelerate>=1.4.0 peft>=0.14.0 \
    huggingface_hub torchmetrics lpips clean-fid kagglehub wandb scikit-learn pandas matplotlib


In [ ]:
!pip install -q -U "torchao>=0.16.0"

# Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, gc, random, warnings, glob
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, DDIMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model
from accelerate import Accelerator
from accelerate.utils import set_seed

import kagglehub
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

@dataclass
class CFG:
    data_root:   str   = ""   # filled by kagglehub
    output_dir:  str   = "/content/checkpoints"
    img_size:    int   = 256
    seasons:     tuple = ("spring", "summer", "fall", "winter")
    val_fraction: float = 0.05
    max_pairs:   int   = None      # None = ALL 600k

    sd_model_id: str   = "runwayml/stable-diffusion-v1-5"
    lora_rank:   int   = 64        # ← was 16, now 64 for Blackwell
    lora_alpha:  int   = 64
    use_dora:    bool  = True

    num_train_steps: int = 200_000
    batch_size:      int   = 16    # ← fits in 96 GB
    grad_accum:      int   = 1     # effective batch = 16
    lr:              float = 1e-4  # slightly higher for big batch
    lr_warmup_steps: int   = 2_000
    mixed_precision: str   = "bf16" # ← Blackwell loves bf16
    gradient_checkpointing: bool = False # ← disable for speed; you have VRAM
    ema_decay:       float = 0.9999

    num_train_timesteps: int = 1000
    ddim_steps:          int = 50

    log_every:  int = 100
    save_every: int = 2_000
    vis_every:  int = 5_000
    seed:       int = 6
    patience:        int = 10_000

cfg = CFG()
set_seed(cfg.seed)
os.makedirs(cfg.output_dir, exist_ok=True)
# print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda}")
# print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Downloading Dataset

In [ ]:

path = kagglehub.dataset_download("shambac/augmented-sentinel-1-2")
print("Dataset path:", path)

cfg.data_root = path
root = Path(cfg.data_root)
print("Seasons:", [d.name for d in root.iterdir() if d.is_dir()])

Dataset path: /root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1
Seasons: ['spring', 'fall', 'summer', 'winter']


Split

In [ ]:
def collect_pairs(root: Path, seasons: tuple, max_pairs: int = None):
    root_str = root.as_posix()
    season_buckets = {s: [] for s in seasons}

    for season in seasons:
        csv_files = list((root / season).glob("*.csv"))
        if not csv_files:
            print(f"[WARN] No CSV for {season}"); continue
        df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
        df["season"] = season
        df["region"] = df["region"].str.strip().str.lower()
        df["s1_fileName"] = (df["s1_fileName"]
                             .str.replace("\\", "/", regex=False)
                             .str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True))
        df["s2_fileName"] = df["s2_fileName"].str.replace("\\", "/", regex=False)
        df["s1"] = root_str + "/" + df["s1_fileName"]
        df["s2"] = root_str + "/" + df["s2_fileName"]
        season_buckets[season] = df[["s1","s2","season","region"]].to_dict("records")
        print(f"  [{season}] {len(season_buckets[season]):,} pairs")

    active = [s for s in seasons if season_buckets[s]]
    if max_pairs is None:
        pairs = [p for s in active for p in season_buckets[s]]
    else:
        per = max_pairs // len(active)
        pairs = []
        for s in active:
            b = season_buckets[s].copy(); random.shuffle(b)
            pairs.extend(b[:per])
    random.shuffle(pairs)
    return pairs


class SAROpticalDataset(Dataset):
    def __init__(self, pairs, img_size=256, augment=True):
        self.pairs, self.img_size, self.augment = pairs, img_size, augment
    def __len__(self): return len(self.pairs)
    def _load(self, path, mode):
        img = Image.open(path).convert(mode)
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        if mode == "L":
            arr = np.stack([arr, arr, arr], axis=2)
        t = torch.from_numpy(arr).permute(2,0,1)
        return t * 2.0 - 1.0
    def __getitem__(self, idx):
        p = self.pairs[idx]
        sar, opt = self._load(p["s1"], "L"), self._load(p["s2"], "RGB")
        if self.augment:
            if random.random() > 0.5: sar = TF.hflip(sar); opt = TF.hflip(opt)
            if random.random() > 0.5: sar = TF.vflip(sar); opt = TF.vflip(opt)
            k = random.randint(0,3)
            if k: sar = torch.rot90(sar, k, [1,2]); opt = torch.rot90(opt, k, [1,2])
        return {"sar": sar, "optical": opt, "season": p["season"], "region": p["region"]}


all_pairs = collect_pairs(root, cfg.seasons, cfg.max_pairs)
print(f"\nTotal: {len(all_pairs):,}")

strata = [f"{p['season']}_{p['region']}" for p in all_pairs]
train_p, val_p = train_test_split(all_pairs, test_size=cfg.val_fraction,
                                  stratify=strata, random_state=cfg.seed)

train_ds = SAROpticalDataset(train_p, cfg.img_size, augment=True)
val_ds   = SAROpticalDataset(val_p,   cfg.img_size, augment=False)

train_dl = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                      num_workers=4, pin_memory=True, drop_last=True,
                      prefetch_factor=2, persistent_workers=True)
val_dl   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                      num_workers=2, pin_memory=True)

print(f"Train {len(train_ds):,} | Val {len(val_ds):,}")

# Model

In [ ]:
vae = AutoencoderKL.from_pretrained(cfg.sd_model_id, subfolder="vae",
                                    torch_dtype=torch.bfloat16)
vae.requires_grad_(False)

unet = UNet2DConditionModel.from_pretrained(cfg.sd_model_id, subfolder="unet",
                                            torch_dtype=torch.float32)

# Expand conv_in 4 → 8
old_conv = unet.conv_in
new_conv = nn.Conv2d(8, old_conv.out_channels, old_conv.kernel_size,
                     old_conv.stride, old_conv.padding,
                     bias=(old_conv.bias is not None))
with torch.no_grad():
    new_conv.weight[:, :4].copy_(old_conv.weight)
    new_conv.weight[:, 4:].zero_()
    if old_conv.bias is not None: new_conv.bias.copy_(old_conv.bias)
unet.conv_in = new_conv
unet.config.in_channels = 8

text_embed_dim = unet.config.cross_attention_dim
null_text_embed = nn.Parameter(torch.randn(1, 77, text_embed_dim) * 0.01)

# DoRA targets: attention + FFN/MLP + block projections
target_modules = [
    "to_q", "to_k", "to_v", "to_out.0",
    "add_q_proj", "add_k_proj", "add_v_proj",
    "ff.net.0.proj", "ff.net.2",      # ← MLP/FFN (you already had these)
    "proj_in", "proj_out",            # ← extra linear projections (new)
]

lora_cfg = LoraConfig(
    r=cfg.lora_rank, lora_alpha=cfg.lora_alpha, use_dora=cfg.use_dora,
    init_lora_weights="gaussian", target_modules=target_modules,
    lora_dropout=0.0, bias="none",
)
unet = get_peft_model(unet, lora_cfg)
unet.conv_in.weight.requires_grad_(True)
if unet.conv_in.bias is not None: unet.conv_in.bias.requires_grad_(True)

trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
total     = sum(p.numel() for p in unet.parameters())
print(f"DoRA rank={cfg.lora_rank} | Trainable {trainable/1e6:.2f}M / {total/1e6:.1f}M ({100*trainable/total:.2f}%)")

noise_scheduler = DDPMScheduler(num_train_timesteps=cfg.num_train_timesteps,
                                beta_schedule="scaled_linear", prediction_type="epsilon")
ddim_scheduler  = DDIMScheduler.from_config(noise_scheduler.config)
ddim_scheduler.set_timesteps(cfg.ddim_steps)

ema_unet = EMAModel(unet.parameters(), decay=cfg.ema_decay)

Accelerator

In [ ]:
trainable_params = list(filter(lambda p: p.requires_grad, unet.parameters()))
trainable_params.append(null_text_embed)

optimizer = torch.optim.AdamW(trainable_params, lr=cfg.lr,
                              betas=(0.9,0.999), weight_decay=1e-2, eps=1e-8)

lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=cfg.lr_warmup_steps * cfg.grad_accum,
    num_training_steps=cfg.num_train_steps * cfg.grad_accum,
)

accelerator = Accelerator(
    mixed_precision=cfg.mixed_precision,
    gradient_accumulation_steps=cfg.grad_accum,
    log_with="wandb",
    project_dir=cfg.output_dir,
)

set_seed(cfg.seed + accelerator.process_index)
unet, vae, optimizer, train_dl, lr_scheduler = accelerator.prepare(
    unet, vae, optimizer, train_dl, lr_scheduler
)

ema_unet.to(accelerator.device)
null_text_embed = null_text_embed.to(accelerator.device)

# --- RESUME LOGIC ---
RESUME_FROM = None
RESUME_STEP = 0
ckpts = sorted(glob.glob(os.path.join(cfg.output_dir, "step_*")))
if ckpts:
    RESUME_FROM = ckpts[-1]
    RESUME_STEP = int(os.path.basename(RESUME_FROM).split("_")[1])
    print(f"Resuming from {RESUME_FROM}")
    accelerator.load_state(RESUME_FROM)
    ema_path = os.path.join(RESUME_FROM, "ema.bin")
    if os.path.exists(ema_path):
        ema_unet.load_state_dict(torch.load(ema_path, map_location=accelerator.device))
    emb_path = os.path.join(RESUME_FROM, "null_embed.pt")
    if os.path.exists(emb_path):
        null_text_embed.data = torch.load(emb_path, map_location=accelerator.device).data
    for _ in range(RESUME_STEP * cfg.grad_accum):
        lr_scheduler.step()
    print(f"Fast-forwarded scheduler to step {RESUME_STEP}")
else:
    print("Starting from scratch.")

if accelerator.is_main_process:
    import wandb
    wandb.login()
    accelerator.init_trackers(
        project_name="sar-optical-blackwell",
        config={"lr": cfg.lr, "batch": cfg.batch_size, "rank": cfg.lora_rank,
                "steps": cfg.num_train_steps, "dataset": len(all_pairs)},
        init_kwargs={"wandb": {"name": f"dora-r{cfg.lora_rank}-blackwell"}}
    )

# Training Loop

In [ ]:
from IPython.display import display
import glob, shutil

# --- Early stopping / best-model tracking ---
BEST_LPIPS = float("inf")
BEST_STEP  = 0
PATIENCE   = 10_000   # stop if no improvement for 10k steps
LAST_IMPROVED = 0

@torch.no_grad()
def encode_to_latent(images):
    images = images.to(dtype=vae.dtype)
    vae_mod = vae.module if hasattr(vae, "module") else vae
    return vae_mod.encode(images).latent_dist.sample() * vae_mod.config.scaling_factor

@torch.no_grad()
def run_validation_and_plot(step, out_dir, n_show=8):
    """Returns average LPIPS on validation set + shows grid inline."""
    os.makedirs(out_dir, exist_ok=True)

    # Pick 2 per season for grid
    selected = []
    for s in cfg.seasons:
        pool = [p for p in val_p if p["season"] == s]
        if pool: selected.extend(random.sample(pool, min(2, len(pool))))
    selected = selected[:n_show]

    vae_mod = vae.module if hasattr(vae, "module") else vae
    unet_eval = accelerator.unwrap_model(unet)
    unet_eval.eval()

    # --- Compute LPIPS on a small val subset (fast) ---
    val_lpips_samples = []
    val_subset = random.sample(val_p, min(32, len(val_p)))
    for pair in val_subset:
        item = SAROpticalDataset([pair], cfg.img_size, False)[0]
        sar = item["sar"].unsqueeze(0).to(accelerator.device, dtype=vae.dtype)
        sar_lat = vae_mod.encode(sar).latent_dist.mean * vae_mod.config.scaling_factor
        lat = torch.randn_like(sar_lat) * ddim_scheduler.init_noise_sigma
        emb = null_text_embed.to(vae.dtype).expand(1,-1,-1)
        ddim_scheduler.set_timesteps(cfg.ddim_steps)
        for t in ddim_scheduler.timesteps:
            mi = torch.cat([lat, sar_lat], 1)
            npred = unet_eval(mi.float(), t.unsqueeze(0).to(accelerator.device),
                              encoder_hidden_states=emb.float()).sample
            lat = ddim_scheduler.step(npred.to(vae.dtype), t, lat).prev_sample
        pred = vae_mod.decode(lat / vae_mod.config.scaling_factor).sample.squeeze(0).clamp(-1,1)
        # quick LPIPS on single image
        p = pred.unsqueeze(0).float().to("cuda")
        g = item["optical"].unsqueeze(0).float().to("cuda")
        val_lpips_samples.append(lpips_fn(p, g).item())

    avg_lpips = np.mean(val_lpips_samples)

    # --- Build visual grid ---
    fig, axes = plt.subplots(len(selected), 3, figsize=(13, 3.8 * len(selected)))
    if len(selected) == 1: axes = axes.reshape(1, -1)

    for i, pair in enumerate(selected):
        item = SAROpticalDataset([pair], cfg.img_size, False)[0]
        sar = item["sar"].unsqueeze(0).to(accelerator.device, dtype=vae.dtype)
        sar_lat = vae_mod.encode(sar).latent_dist.mean * vae_mod.config.scaling_factor
        lat = torch.randn_like(sar_lat) * ddim_scheduler.init_noise_sigma
        emb = null_text_embed.to(vae.dtype).expand(1,-1,-1)
        ddim_scheduler.set_timesteps(cfg.ddim_steps)
        for t in ddim_scheduler.timesteps:
            mi = torch.cat([lat, sar_lat], 1)
            npred = unet_eval(mi.float(), t.unsqueeze(0).to(accelerator.device),
                              encoder_hidden_states=emb.float()).sample
            lat = ddim_scheduler.step(npred.to(vae.dtype), t, lat).prev_sample
        pred = vae_mod.decode(lat / vae_mod.config.scaling_factor).sample.squeeze(0).clamp(-1,1).float().cpu()

        def dn(t): return ((t.clamp(-1,1)+1)/2).permute(1,2,0).numpy()

        axes[i,0].imshow(dn(item["sar"])[:,:,0], cmap="gray")
        axes[i,0].set_title(f"SAR | {pair['season']} | {pair['region']}", fontsize=8)
        axes[i,0].axis("off")

        axes[i,1].imshow(dn(pred))
        axes[i,1].set_title(f"Predicted (step {step})", fontsize=8)
        axes[i,1].axis("off")

        axes[i,2].imshow(dn(item["optical"]))
        axes[i,2].set_title("Ground Truth", fontsize=8)
        axes[i,2].axis("off")

    plt.suptitle(f"SAR → Optical | Step {step} | Val LPIPS {avg_lpips:.4f} (↓ better)", fontsize=12, y=1.01)
    plt.tight_layout()

    # --- INLINE DISPLAY FOR COLAB ---
    display(fig)

    p = os.path.join(out_dir, f"val_step_{step:06d}.png")
    plt.savefig(p, dpi=120, bbox_inches="tight")
    plt.close(fig)

    unet_eval.train()
    return avg_lpips, p


# --- Training Loop ---
global_step = RESUME_STEP
unet.train()

# LPIPS for validation (declare here so validation can use it)
lpips_fn = lpips_lib.LPIPS(net="alex").to(accelerator.device)

pbar = tqdm(total=cfg.num_train_steps, initial=global_step,
            disable=not accelerator.is_main_process)

while global_step < cfg.num_train_steps:
    for batch in train_dl:
        with accelerator.accumulate(unet):
            sar = batch["sar"].to(accelerator.device)
            opt = batch["optical"].to(accelerator.device)

            opt_lat = encode_to_latent(opt)
            sar_lat = encode_to_latent(sar)
            noise   = torch.randn_like(opt_lat)
            B = opt_lat.shape[0]
            ts = torch.randint(0, cfg.num_train_timesteps, (B,),
                               device=accelerator.device, dtype=torch.long)

            noisy = noise_scheduler.add_noise(opt_lat, noise, ts)
            model_in = torch.cat([noisy, sar_lat], dim=1)
            enc = null_text_embed.to(opt_lat.dtype).expand(B, -1, -1)

            noise_pred = unet(model_in, ts, encoder_hidden_states=enc).sample
            loss = F.mse_loss(noise_pred.float(), noise.float())

            accelerator.backward(loss)
            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(accelerator.unwrap_model(unet).parameters(), 1.0)

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        if accelerator.sync_gradients:
            ema_unet.step(accelerator.unwrap_model(unet).parameters())
            global_step += 1
            lr_now = lr_scheduler.get_last_lr()[0]
            pbar.update(1)
            pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "lr": f"{lr_now:.1e}",
                "best": f"{BEST_STEP}",
                "best_l": f"{BEST_LPIPS:.4f}"
            })

            if global_step % cfg.log_every == 0 and accelerator.is_main_process:
                accelerator.log({
                    "train/loss": loss.item(),
                    "train/lr": lr_now,
                    "train/best_lpips": BEST_LPIPS,
                    "train/best_step": BEST_STEP,
                }, step=global_step)

            # --- VALIDATION + INLINE PLOT ---
            if global_step % cfg.vis_every == 0 and accelerator.is_main_process:
                tqdm.write(f"\n>>> Running validation at step {global_step}...")
                ema_unet.copy_to(accelerator.unwrap_model(unet).parameters())
                val_lpips, grid_path = run_validation_and_plot(
                    global_step,
                    os.path.join(cfg.output_dir, "val_grids")
                )
                tqdm.write(f">>> Val LPIPS: {val_lpips:.4f} | Grid: {grid_path}\n")

                # --- BEST MODEL SAVE ---
                if val_lpips < BEST_LPIPS:
                    BEST_LPIPS = val_lpips
                    BEST_STEP = global_step
                    LAST_IMPROVED = global_step

                    best_dir = os.path.join(cfg.output_dir, "best_model")
                    if os.path.exists(best_dir):
                        shutil.rmtree(best_dir)
                    os.makedirs(best_dir, exist_ok=True)

                    uw = accelerator.unwrap_model(unet)
                    uw.save_pretrained(os.path.join(best_dir, "unet_adapter"))
                    torch.save(null_text_embed, os.path.join(best_dir, "null_embed.pt"))
                    torch.save(ema_unet.state_dict(), os.path.join(best_dir, "ema.bin"))

                    # Save metadata
                    with open(os.path.join(best_dir, "info.txt"), "w") as f:
                        f.write(f"step: {global_step}\nlpips: {val_lpips:.6f}\n")

                    tqdm.write(f"*** NEW BEST MODEL *** LPIPS {val_lpips:.4f} -> {best_dir}")
                else:
                    # Early stopping check
                    if global_step - LAST_IMPROVED > PATIENCE:
                        tqdm.write(f"\n!!! EARLY STOPPING !!! No improvement for {PATIENCE} steps. Best was step {BEST_STEP}.")
                        global_step = cfg.num_train_steps  # break outer loop
                        break

            # --- REGULAR RESUME CHECKPOINT (full state) ---
            if global_step % cfg.save_every == 0 and accelerator.is_main_process:
                cdir = os.path.join(cfg.output_dir, f"step_{global_step}")
                os.makedirs(cdir, exist_ok=True)
                accelerator.save_state(cdir)
                torch.save(ema_unet.state_dict(), os.path.join(cdir, "ema.bin"))
                torch.save(null_text_embed, os.path.join(cdir, "null_embed.pt"))
                tqdm.write(f"Resume checkpoint -> {cdir}")

                # Cleanup old resume checkpoints (keep last 3)
                all_ckpts = sorted(glob.glob(os.path.join(cfg.output_dir, "step_*")))
                for old in all_ckpts[:-3]:
                    shutil.rmtree(old)
                    tqdm.write(f"Cleaned old checkpoint: {old}")

        if global_step >= cfg.num_train_steps:
            break

    gc.collect()
    torch.cuda.empty_cache()
    if global_step >= cfg.num_train_steps:
        break

pbar.close()

# --- FINAL SAVE ---
if accelerator.is_main_process:
    final_dir = os.path.join(cfg.output_dir, f"final_step_{global_step}")
    os.makedirs(final_dir, exist_ok=True)
    uw = accelerator.unwrap_model(unet)

    # Save final EMA state
    ema_unet.copy_to(uw.parameters())
    uw.save_pretrained(os.path.join(final_dir, "unet_adapter"))
    merged = uw.merge_and_unload()
    merged.save_pretrained(os.path.join(final_dir, "unet_full"))
    torch.save(null_text_embed, os.path.join(final_dir, "null_embed.pt"))
    torch.save(ema_unet.state_dict(), os.path.join(final_dir, "ema.bin"))

    print(f"\nTraining complete. Final -> {final_dir}")
    print(f"BEST MODEL was step {BEST_STEP} with LPIPS {BEST_LPIPS:.4f}")
    print(f"Located at: {os.path.join(cfg.output_dir, 'best_model')}")

accelerator.end_training()

# Evaluation

In [ ]:
import glob
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
import lpips as lpips_lib

# Find latest final checkpoint
final_dirs = sorted(glob.glob(os.path.join(cfg.output_dir, "final_step_*")))
if not final_dirs:
    raise FileNotFoundError("No final checkpoint found. Train first or set path manually.")

ckpt = final_dirs[-1]
print(f"Evaluating from: {ckpt}")

# Load base + adapter
eval_vae = AutoencoderKL.from_pretrained(cfg.sd_model_id, subfolder="vae",
                                         torch_dtype=torch.bfloat16).to("cuda").eval()
eval_vae.requires_grad_(False)

base_unet = UNet2DConditionModel.from_pretrained(cfg.sd_model_id, subfolder="unet",
                                                  torch_dtype=torch.bfloat16)
old = base_unet.conv_in
new = nn.Conv2d(8, old.out_channels, old.kernel_size, old.stride, old.padding,
                bias=(old.bias is not None)).to(torch.bfloat16)
with torch.no_grad():
    new.weight[:,:4].copy_(old.weight); new.weight[:,4:].zero_()
    if old.bias is not None: new.bias.copy_(old.bias)
base_unet.conv_in = new; base_unet.config.in_channels = 8

base_unet.load_adapter(os.path.join(ckpt, "unet_adapter"))
eval_unet = base_unet.to("cuda").eval()

eval_null = torch.load(os.path.join(ckpt, "null_embed.pt"), map_location="cuda")
if isinstance(eval_null, nn.Parameter): eval_null = eval_null.data
eval_null = eval_null.bfloat16().unsqueeze(0) if eval_null.dim()==2 else eval_null.bfloat16()

ddim = DDIMScheduler.from_config(noise_scheduler.config)
ddim.set_timesteps(50)

psnr_fn = PeakSignalNoiseRatio(data_range=2.0).to("cuda")
ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to("cuda")
lpips_fn = lpips_lib.LPIPS(net="alex").to("cuda")

@torch.no_grad()
def infer(sar_tensor):
    sar = sar_tensor.unsqueeze(0).to("cuda", dtype=torch.bfloat16)
    s_lat = eval_vae.encode(sar).latent_dist.mean * eval_vae.config.scaling_factor
    lat = torch.randn_like(s_lat) * ddim.init_noise_sigma
    emb = eval_null.expand(1,-1,-1)
    ddim.set_timesteps(50)
    for t in ddim.timesteps:
        mi = torch.cat([lat, s_lat], 1)
        npred = eval_unet(mi, t.unsqueeze(0).to("cuda"), encoder_hidden_states=emb).sample
        lat = ddim.step(npred, t, lat).prev_sample
    img = eval_vae.decode(lat/eval_vae.config.scaling_factor).sample
    return img.squeeze(0).clamp(-1,1).float()

# Run on 200 random val patches
n_eval = min(200, len(val_p))
subset = random.sample(val_p, n_eval)
results = {"psnr": [], "ssim": [], "lpips": []}

for pair in tqdm(subset, desc="Eval"):
    item = SAROpticalDataset([pair], cfg.img_size, False)[0]
    pred = infer(item["sar"])
    p = pred.unsqueeze(0).to("cuda")
    g = item["optical"].unsqueeze(0).to("cuda")
    results["psnr"].append(psnr_fn(p,g).item())
    results["ssim"].append(ssim_fn(p,g).item())
    results["lpips"].append(lpips_fn(p,g).item())

print(f"\nPSNR  : {np.mean(results['psnr']):.3f} ± {np.std(results['psnr']):.3f} dB")
print(f"SSIM  : {np.mean(results['ssim']):.4f} ± {np.std(results['ssim']):.4f}")
print(f"LPIPS : {np.mean(results['lpips']):.4f} ± {np.std(results['lpips']):.4f}  (↓ better)")

# Experiment # 7

In [ ]:
%%writefile /content/eval_resshift.py
import os, gc, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from diffusers import UNet2DConditionModel, AutoencoderKL
from diffusers.models.attention_processor import AttnProcessor2_0
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
import lpips as lpips_lib
import safetensors.torch as st
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split

# ── Config ────────────────────────────────────────────────────────────────────
HF_REPO_ID    = "AliMusaRizvi/sar-to-optical-diffusion"
SD_MODEL_ID   = "runwayml/stable-diffusion-v1-5"
DATA_ROOT     = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR    = "/content/eval_resshift"
IMG_SIZE      = 256
SEASONS       = ("spring", "summer", "fall", "winter")
TERRAIN_TYPES = ("tropical", "temperate", "arctic", "arid", "coastal", "urban")
VAL_FRACTION  = 0.05
SEED          = 42
N_EVAL        = 300
FIXED_SEED    = 999

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"Repo   : https://huggingface.co/{HF_REPO_ID}")


# ── ResShift Scheduler ────────────────────────────────────────────────────────
class ResShiftScheduler:
    def __init__(self, num_timesteps=15, kappa=2.0, eta=0.5,
                 schedule="exponential", device="cuda"):
        self.T      = num_timesteps
        self.device = device
        if schedule == "exponential":
            t     = torch.arange(1, num_timesteps + 1, dtype=torch.float32)
            eta_t = (torch.exp(t / num_timesteps * np.log(kappa + 1)) - 1) / kappa
        else:
            eta_t = torch.linspace(0, 1, num_timesteps + 1)[1:]
        self.eta_t     = eta_t.to(device)
        t_norm         = torch.arange(1, num_timesteps + 1, dtype=torch.float32) / num_timesteps
        self.sigma_t   = (eta * t_norm.sqrt()).to(device)
        self.timesteps = torch.arange(num_timesteps - 1, -1, -1, device=device)

    def get_x0_from_pred(self, x_t, y, t, x0_pred):
        if t.item() == 0:
            return x0_pred
        eta_prev = self.eta_t[t - 1].view(-1, 1, 1, 1)
        return x0_pred + eta_prev * (y - x0_pred)


# ── Helpers ───────────────────────────────────────────────────────────────────
def map_region_to_terrain(region):
    region = region.lower()
    for k, v in {
        "tropical": "tropical", "temperate": "temperate",
        "arctic":   "arctic",   "arid":      "arid",
        "desert":   "arid",     "coastal":   "coastal",
        "ocean":    "coastal",  "urban":     "urban",
        "city":     "urban",
    }.items():
        if k in region: return v
    return "temperate"


def denorm(t):
    return ((t.clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy()


# ── Dataset ───────────────────────────────────────────────────────────────────
def collect_pairs(root, seasons):
    root_str = Path(root).as_posix()
    buckets  = {s: [] for s in seasons}
    for season in seasons:
        csvs = list((Path(root) / season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df["season"] = season
        df["region"] = df["region"].str.strip().str.lower()
        df["s1_fileName"] = (df["s1_fileName"]
                             .str.replace("\\", "/", regex=False)
                             .str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True))
        df["s2_fileName"] = df["s2_fileName"].str.replace("\\", "/", regex=False)
        df["s1"] = root_str + "/" + df["s1_fileName"]
        df["s2"] = root_str + "/" + df["s2_fileName"]
        buckets[season] = df[["s1","s2","season","region"]].to_dict("records")
    return [p for s in seasons if buckets[s] for p in buckets[s]]


class SARDataset(Dataset):
    def __init__(self, pairs, img_size=512):
        self.pairs = pairs; self.img_size = img_size

    def __len__(self): return len(self.pairs)

    def _load(self, path, mode):
        img = Image.open(path).convert(mode)
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        if mode == "L": arr = np.stack([arr, arr, arr], axis=2)
        return torch.from_numpy(arr).permute(2, 0, 1) * 2.0 - 1.0

    def __getitem__(self, idx):
        p = self.pairs[idx]
        return {"sar": self._load(p["s1"], "L"),
                "optical": self._load(p["s2"], "RGB"),
                "season": p["season"], "region": p["region"]}


# ── Step 1: Download from HF ──────────────────────────────────────────────────
print("\n" + "="*60)
print("Step 1: Downloading ResShift checkpoint from HuggingFace")
print("="*60)

def try_download(folder, filename):
    try:
        return hf_hub_download(repo_id=HF_REPO_ID, filename=f"{folder}/{filename}")
    except Exception:
        return None

# Search for weights across possible checkpoint tags
ckpt_tag   = None
model_path = None
for tag in [ "resshift_final"]:
    p = try_download(tag, "model/model.safetensors")
    if p:
        ckpt_tag = tag; model_path = p
        print(f"  Weights found : {tag}/model/model.safetensors")
        break
    p = try_download(tag, "model.safetensors")
    if p:
        ckpt_tag = tag; model_path = p
        print(f"  Weights found : {tag}/model.safetensors")
        break

if model_path is None:
    raise FileNotFoundError(
        "No ResShift weights found on HF. "
        "Expected: resshift_best/model/model.safetensors"
    )

embed_cache_path = hf_hub_download(repo_id=HF_REPO_ID,
                                    filename=f"{ckpt_tag}/embed_cache.pt")
rs_config_path   = hf_hub_download(repo_id=HF_REPO_ID,
                                    filename=f"{ckpt_tag}/resshift_config.pt")

try:
    step_path = hf_hub_download(repo_id=HF_REPO_ID,
                                 filename=f"{ckpt_tag}/step.txt")
    with open(step_path) as f:
        total_steps = f.read().strip()
except Exception:
    total_steps = "unknown"

print(f"  Checkpoint    : {ckpt_tag}")
print(f"  Total steps   : {total_steps}")


# ── Step 2: ResShift config & scheduler ───────────────────────────────────────
print("\n" + "="*60)
print("Step 2: Loading ResShift config")
print("="*60)

rs_cfg        = torch.load(rs_config_path, map_location="cpu", weights_only=False)
NUM_TIMESTEPS = rs_cfg.get("T",        15)
KAPPA         = rs_cfg.get("kappa",   2.0)
ETA           = rs_cfg.get("eta",     0.5)
SCHEDULE      = rs_cfg.get("schedule", "exponential")
print(f"  T={NUM_TIMESTEPS}  kappa={KAPPA}  eta={ETA}  schedule={SCHEDULE}")

rs_scheduler = ResShiftScheduler(
    num_timesteps=NUM_TIMESTEPS, kappa=KAPPA,
    eta=ETA, schedule=SCHEDULE, device=device,
)


# ── Step 3: Load UNet ─────────────────────────────────────────────────────────
print("\n" + "="*60)
print("Step 3: Loading ResShift UNet (8-channel conv_in)")
print("="*60)

unet     = UNet2DConditionModel.from_pretrained(
    SD_MODEL_ID, subfolder="unet", torch_dtype=torch.bfloat16)
old_conv = unet.conv_in
conv_8ch = nn.Conv2d(
    8, old_conv.out_channels, old_conv.kernel_size,
    old_conv.stride, old_conv.padding,
    bias=(old_conv.bias is not None),
).to(torch.bfloat16)
with torch.no_grad():
    conv_8ch.weight[:, :4].copy_(old_conv.weight)
    conv_8ch.weight[:, 4:].zero_()
    if old_conv.bias is not None: conv_8ch.bias.copy_(old_conv.bias)
unet.conv_in = conv_8ch
unet.config.in_channels = 8

state_dict = st.load_file(model_path, device="cpu")
missing, unexpected = unet.load_state_dict(state_dict, strict=False)
print(f"  Missing: {len(missing)}  Unexpected: {len(unexpected)}")

unet.set_attn_processor(AttnProcessor2_0())
unet.requires_grad_(False)
unet = unet.to(device).eval()
print(f"  UNet ready: {sum(p.numel() for p in unet.parameters())/1e6:.0f}M params")


# ── Step 4: VAE + embedding cache ─────────────────────────────────────────────
print("\n" + "="*60)
print("Step 4: Loading VAE + embedding cache")
print("="*60)

vae = AutoencoderKL.from_pretrained(
    SD_MODEL_ID, subfolder="vae", torch_dtype=torch.bfloat16).to(device).eval()
vae.requires_grad_(False)

embed_cache = {
    k: v.to(device)
    for k, v in torch.load(embed_cache_path, map_location=device,
                            weights_only=False).items()
}
print(f"  Embed cache: {len(embed_cache)} embeddings")


# ── Step 5: Validation split ──────────────────────────────────────────────────
print("\n" + "="*60)
print("Step 5: Building validation split")
print("="*60)

all_pairs = collect_pairs(DATA_ROOT, SEASONS)
_, val_pairs = train_test_split(
    all_pairs, test_size=VAL_FRACTION,
    stratify=[f"{p['season']}_{p['region']}" for p in all_pairs],
    random_state=SEED,
)
print(f"  Validation set: {len(val_pairs):,} pairs")


# ── Inference ─────────────────────────────────────────────────────────────────
def get_embed(season, region):
    t   = map_region_to_terrain(region)
    key = f"{season}_{t}"
    return embed_cache.get(key, embed_cache.get(f"season_{season}",
                                                 embed_cache["null"]))


@torch.no_grad()
def translate(sar_tensor, season, region, num_steps=None):
    sched   = rs_scheduler if num_steps is None else ResShiftScheduler(
        num_timesteps=num_steps, kappa=KAPPA, eta=ETA,
        schedule=SCHEDULE, device=device,
    )
    sar     = sar_tensor.unsqueeze(0).to(device, dtype=torch.bfloat16)
    sar_lat = vae.encode(sar).latent_dist.mean * vae.config.scaling_factor
    x_t     = sar_lat + sched.sigma_t[-1] * torch.randn_like(sar_lat)
    embed   = get_embed(season, region).to(torch.bfloat16).expand(1, -1, -1)

    for t_idx in sched.timesteps:
        t_tensor = t_idx.unsqueeze(0).to(device)
        model_in = torch.cat([x_t, sar_lat], dim=1).to(torch.bfloat16)
        x0_pred  = unet(model_in, t_tensor,
                        encoder_hidden_states=embed).sample.to(torch.bfloat16)
        x_t      = sched.get_x0_from_pred(x_t, sar_lat, t_idx, x0_pred)

    image = vae.decode(
        (x_t / vae.config.scaling_factor).to(vae.dtype)
    ).sample
    return image.squeeze(0).clamp(-1, 1).float().cpu()


# ── Step 6: Quantitative metrics ─────────────────────────────────────────────
print("\n" + "="*60)
print(f"Step 6: Quantitative metrics ({N_EVAL} patches)")
print("="*60)

psnr_fn  = PeakSignalNoiseRatio(data_range=2.0).to(device)
ssim_fn  = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)
lpips_fn = lpips_lib.LPIPS(net="alex").to(device)

eval_subset     = random.sample(val_pairs, min(N_EVAL, len(val_pairs)))
season_metrics  = {s: {"psnr":[], "ssim":[], "lpips":[]} for s in SEASONS}
terrain_metrics = {t: {"psnr":[], "ssim":[], "lpips":[]} for t in TERRAIN_TYPES}
all_psnr, all_ssim, all_lpips = [], [], []

for pair in tqdm(eval_subset, desc="Computing metrics"):
    ds   = SARDataset([pair], img_size=IMG_SIZE)
    item = ds[0]
    pred = translate(item["sar"], pair["season"], pair["region"])
    p    = pred.unsqueeze(0).to(device).float()
    g    = item["optical"].unsqueeze(0).to(device).float()

    pv  = psnr_fn(p, g).item()
    sv  = ssim_fn(p, g).item()
    lv  = lpips_fn(p, g).item()
    all_psnr.append(pv); all_ssim.append(sv); all_lpips.append(lv)

    s = pair["season"]; t = map_region_to_terrain(pair["region"])
    for key, val in zip(["psnr","ssim","lpips"], [pv, sv, lv]):
        if s in season_metrics:  season_metrics[s][key].append(val)
        if t in terrain_metrics: terrain_metrics[t][key].append(val)

print(f"\n{'─'*58}")
print(f"OVERALL ({N_EVAL} patches) — ResShift {NUM_TIMESTEPS}-step | {total_steps} total steps")
print(f"  PSNR  : {np.mean(all_psnr):.3f} ± {np.std(all_psnr):.3f} dB")
print(f"  SSIM  : {np.mean(all_ssim):.4f} ± {np.std(all_ssim):.4f}")
print(f"  LPIPS : {np.mean(all_lpips):.4f} ± {np.std(all_lpips):.4f}  (↓ better)")
print(f"{'─'*58}")
print("BY SEASON:")
for s in SEASONS:
    m = season_metrics[s]
    if not m["psnr"]: continue
    print(f"  {s:<8}  PSNR {np.mean(m['psnr']):.3f}  "
          f"SSIM {np.mean(m['ssim']):.4f}  LPIPS {np.mean(m['lpips']):.4f}")
print(f"{'─'*58}")
print("BY TERRAIN:")
for t in TERRAIN_TYPES:
    m = terrain_metrics[t]
    if not m["psnr"]: continue
    print(f"  {t:<12}  PSNR {np.mean(m['psnr']):.3f}  "
          f"SSIM {np.mean(m['ssim']):.4f}  LPIPS {np.mean(m['lpips']):.4f}")


# ── Step 7: Visual comparison grid ───────────────────────────────────────────
print("\n" + "="*60)
print("Step 7: Visual comparison grid (2 per season, fixed seed)")
print("="*60)

rng      = random.Random(FIXED_SEED)
selected = []
for s in SEASONS:
    pool = [p for p in val_pairs if p["season"] == s]
    if pool: selected.extend(rng.sample(pool, min(2, len(pool))))

n   = len(selected)
fig = plt.figure(figsize=(15, 4 * n))
gs  = gridspec.GridSpec(n, 3, figure=fig, hspace=0.35, wspace=0.05)

for i, pair in enumerate(tqdm(selected, desc="Visual grid")):
    ds   = SARDataset([pair], img_size=IMG_SIZE)
    item = ds[0]
    pred = translate(item["sar"], pair["season"], pair["region"])

    p   = pred.unsqueeze(0).to(device).float()
    g   = item["optical"].unsqueeze(0).to(device).float()
    pv  = psnr_fn(p, g).item()
    sv  = ssim_fn(p, g).item()
    lv  = lpips_fn(p, g).item()
    ter = map_region_to_terrain(pair["region"])

    ax0 = fig.add_subplot(gs[i, 0])
    ax1 = fig.add_subplot(gs[i, 1])
    ax2 = fig.add_subplot(gs[i, 2])

    ax0.imshow(denorm(item["sar"]))
    ax0.set_title(f"SAR Input\n{pair['season']} | {ter}", fontsize=9, fontweight="bold")
    ax0.axis("off")

    ax1.imshow(denorm(pred))
    ax1.set_title(
        f"ResShift ({NUM_TIMESTEPS} steps)\n"
        f"PSNR {pv:.2f} dB  SSIM {sv:.3f}  LPIPS {lv:.3f}",
        fontsize=8, fontweight="bold",
    )
    ax1.axis("off")

    ax2.imshow(denorm(item["optical"]))
    ax2.set_title("Ground Truth (Sentinel-2)", fontsize=9, fontweight="bold")
    ax2.axis("off")

plt.suptitle(
    f"SAR-INTEL ResShift — {ckpt_tag} — {total_steps} steps",
    fontsize=13, fontweight="bold", y=1.01,
)
plt.tight_layout()
grid_path = os.path.join(OUTPUT_DIR, "visual_grid.png")
plt.savefig(grid_path, dpi=130, bbox_inches="tight")
plt.show(); plt.close()
print(f"  Saved -> {grid_path}")


# ── Step 8: Steps ablation ────────────────────────────────────────────────────
print("\n" + "="*60)
print("Step 8: Inference steps ablation (5 / 10 / 15 / 25)")
print("="*60)

ablation_subset = rng.sample(val_pairs, 20)
step_options    = [5, 10, 15, 25]
step_results    = {s: {"psnr":[], "ssim":[], "lpips":[]} for s in step_options}

for pair in tqdm(ablation_subset, desc="Steps ablation"):
    ds   = SARDataset([pair], img_size=IMG_SIZE)
    item = ds[0]
    for ns in step_options:
        pred = translate(item["sar"], pair["season"], pair["region"], num_steps=ns)
        p    = pred.unsqueeze(0).to(device).float()
        g    = item["optical"].unsqueeze(0).to(device).float()
        step_results[ns]["psnr"].append(psnr_fn(p, g).item())
        step_results[ns]["ssim"].append(ssim_fn(p, g).item())
        step_results[ns]["lpips"].append(lpips_fn(p, g).item())

best_ns = max(step_options, key=lambda s: np.mean(step_results[s]["psnr"]))
print(f"\n{'Steps':<8} {'PSNR':>8} {'SSIM':>8} {'LPIPS':>8}")
print("─" * 40)
for ns in step_options:
    m   = step_results[ns]
    tag = "  ← BEST" if ns == best_ns else ""
    print(f"{ns:<8} {np.mean(m['psnr']):>8.3f} {np.mean(m['ssim']):>8.4f} "
          f"{np.mean(m['lpips']):>8.4f}{tag}")


# ── Step 9: Distribution plots ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, vals, title, xlabel, color in zip(
    axes,
    [all_psnr, all_ssim, all_lpips],
    ["PSNR Distribution", "SSIM Distribution", "LPIPS Distribution (↓ better)"],
    ["dB", "SSIM", "LPIPS"],
    ["steelblue", "seagreen", "coral"],
):
    ax.hist(vals, bins=35, color=color, edgecolor="white", alpha=0.85)
    ax.axvline(np.mean(vals), color="red", linestyle="--", lw=2,
               label=f"Mean {np.mean(vals):.3f}")
    ax.axvline(np.median(vals), color="orange", linestyle=":", lw=2,
               label=f"Median {np.median(vals):.3f}")
    ax.set_title(title, fontweight="bold"); ax.set_xlabel(xlabel); ax.legend()
plt.suptitle("ResShift Metric Distributions", fontsize=12, fontweight="bold")
dist_path = os.path.join(OUTPUT_DIR, "distributions.png")
plt.savefig(dist_path, dpi=130, bbox_inches="tight")
plt.show(); plt.close()


# ── Step 10: Per-season bars ──────────────────────────────────────────────────
seasons_ok = [s for s in SEASONS if season_metrics[s]["psnr"]]
x  = np.arange(len(seasons_ok))
cl = ["#4CAF50", "#FF9800", "#F44336", "#2196F3"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, key, title, yl in zip(
    axes,
    ["psnr", "ssim", "lpips"],
    ["PSNR by Season (↑ better)", "SSIM by Season (↑ better)", "LPIPS by Season (↓ better)"],
    ["dB", "SSIM", "LPIPS"],
):
    vals = [np.mean(season_metrics[s][key]) for s in seasons_ok]
    stds = [np.std(season_metrics[s][key])  for s in seasons_ok]
    bars = ax.bar(x, vals, color=cl[:len(seasons_ok)], edgecolor="white",
                  yerr=stds, capsize=4)
    ax.set_xticks(x); ax.set_xticklabels(seasons_ok)
    ax.set_title(title, fontweight="bold"); ax.set_ylabel(yl)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(stds) * 0.05,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
plt.suptitle("Per-Season Performance — ResShift", fontsize=12, fontweight="bold")
plt.tight_layout()
season_path = os.path.join(OUTPUT_DIR, "season_bars.png")
plt.savefig(season_path, dpi=130, bbox_inches="tight")
plt.show(); plt.close()


# ── Step 11: Per-terrain bars ─────────────────────────────────────────────────
terrains_ok = [t for t in TERRAIN_TYPES if terrain_metrics[t]["psnr"]]
x   = np.arange(len(terrains_ok))
clt = ["#9C27B0","#00BCD4","#8BC34A","#FF5722","#607D8B","#FFC107"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, key, title, yl in zip(
    axes,
    ["psnr", "ssim", "lpips"],
    ["PSNR by Terrain (↑ better)", "SSIM by Terrain (↑ better)", "LPIPS by Terrain (↓ better)"],
    ["dB", "SSIM", "LPIPS"],
):
    vals = [np.mean(terrain_metrics[t][key]) for t in terrains_ok]
    stds = [np.std(terrain_metrics[t][key])  for t in terrains_ok]
    bars = ax.bar(x, vals, color=clt[:len(terrains_ok)], edgecolor="white",
                  yerr=stds, capsize=4)
    ax.set_xticks(x); ax.set_xticklabels(terrains_ok, rotation=30, ha="right")
    ax.set_title(title, fontweight="bold"); ax.set_ylabel(yl)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(stds) * 0.05,
                f"{v:.3f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
plt.suptitle("Per-Terrain Performance — ResShift", fontsize=12, fontweight="bold")
plt.tight_layout()
terrain_path = os.path.join(OUTPUT_DIR, "terrain_bars.png")
plt.savefig(terrain_path, dpi=130, bbox_inches="tight")
plt.show(); plt.close()


# ── Step 12: Verdict ──────────────────────────────────────────────────────────
mean_psnr  = np.mean(all_psnr)
mean_ssim  = np.mean(all_ssim)
mean_lpips = np.mean(all_lpips)

print("\n" + "="*60)
print("FINAL VERDICT")
print("="*60)
print(f"  Checkpoint  : {ckpt_tag}  ({total_steps} total steps)")
print(f"  Inference   : {NUM_TIMESTEPS} ResShift steps (deterministic)")
print(f"\n  PSNR  : {mean_psnr:.3f} dB")
print(f"  SSIM  : {mean_ssim:.4f}")
print(f"  LPIPS : {mean_lpips:.4f}")
print(f"\n  Baselines:")
print(f"    Phase A (156k, DDPM 50-step)    : ~10.6 dB / 0.055 / 0.772")
print(f"    ResShift 180k steps             : ~12.5 dB / 0.154 / 1.006")
print(f"    SOTA (color-supervised diffusion): ~19.7 dB / 0.312")
print(f"\n  Improvement vs Phase A : {mean_psnr - 10.6:+.2f} dB PSNR")
print(f"  Improvement vs 180k    : {mean_psnr - 12.5:+.2f} dB PSNR")

if mean_psnr >= 18.0:
    verdict = "EXCELLENT — Target reached."; action = "Deploy. Training complete."
elif mean_psnr >= 15.0:
    verdict = "GOOD — Strong improvement.";  action = "Run 50k more steps at LR=2e-6."
elif mean_psnr >= 13.0:
    verdict = "IMPROVING — Converging steadily."; action = "Run 50k more steps at LR=5e-6."
else:
    verdict = "EARLY — More training needed."; action = "Run 100k more steps."

print(f"\n  Verdict : {verdict}")
print(f"  Action  : {action}")
print(f"\n  Plots saved -> {OUTPUT_DIR}")
print("="*60)

gc.collect(); torch.cuda.empty_cache()

Writing /content/eval_resshift.py


In [ ]:
!pip install -q lpips torchmetrics
!python /content/eval_resshift.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Device : cuda
Repo   : https://huggingface.co/AliMusaRizvi/sar-to-optical-diffusion

Step 1: Downloading ResShift checkpoint from HuggingFace
resshift_final/model/model.safetensors: 100% 

In [ ]:
%%writefile /content/stage1_detail_vae.py
"""
SAR-INTEL Stage 1 — Detail-Aligned VAE (DA-VAE applied to Sentinel-2 optical)
Goal: lift the VAE reconstruction ceiling that caps the whole SAR->optical field.

Method (faithful to DA-VAE, CVPR 2026):
  - Base latent z (4ch): FROZEN SD-1.5 VAE encoder. This is the latent the
    286k-step bridge UNet already speaks — we must NOT disturb it.
  - Detail latent z_d (12ch): NEW trainable detail encoder. Total = 16 channels.
  - Decoder reconstructs from [z, z_d]; decoder.conv_in widened 4->16,
    extra channels ZERO-INIT (warm start: identical to vanilla at step 0).
  - Alignment loss: project z_d -> 4ch by grouped averaging, L2 to z.
    (DA-VAE ablation: removing this collapses quality.)

Stage 1 trains: detail_encoder + decoder. Frozen: encoder + quant_conv.
Output: a VAE whose 4 base channels are byte-identical in meaning to SD-1.5,
plus 12 detail channels that recover high-frequency optical detail.
"""
import os, gc, random, warnings, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from diffusers import AutoencoderKL
from accelerate import Accelerator
from accelerate.utils import set_seed
from tqdm.auto import tqdm
from huggingface_hub import HfApi
import lpips as lpips_lib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ── Config ────────────────────────────────────────────────────────────────────
DATA_ROOT       = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR      = "/content/drive/MyDrive/stage1_detail_vae"
SD_MODEL_ID     = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID      = "AliMusaRizvi/sar-to-optical-diffusion"
HF_WRITE_TOKEN  = os.environ.get("HF_WRITE_TOKEN", "")

IMG_SIZE        = 512
SEASONS         = ("spring", "summer", "fall", "winter")
VAL_FRACTION    = 0.05
SEED            = 42

BASE_CHANNELS   = 4          # SD-1.5 VAE latent channels (frozen, the UNet speaks this)
DETAIL_CHANNELS = 12         # added detail channels -> total 16
TOTAL_CHANNELS  = BASE_CHANNELS + DETAIL_CHANNELS

# Stage 1 is cheap & falsifiable — keep it short, eval often, stop early if ceiling lifts
NUM_TRAIN_STEPS = 40_000
BATCH_SIZE      = 4
GRAD_ACCUM      = 2          # effective batch 8
LR              = 1e-4
LR_WARMUP       = 500
ALIGN_WEIGHT    = 0.5        # DA-VAE default; ablated as the sweet spot
LPIPS_WEIGHT    = 1.0
L1_WEIGHT       = 1.0

CEILING_EVAL_EVERY = 5_000   # measure recon PSNR vs vanilla at these intervals
N_CEILING_EVAL     = 200     # patches for the ceiling measurement
VIS_EVERY          = 5_000
SAVE_EVERY         = 10_000
FIXED_VAL_SEED     = 999

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ── Data (optical only needed for reconstruction) ─────────────────────────────
def collect_pairs(root, seasons):
    root_str = Path(root).as_posix()
    buckets  = {s: [] for s in seasons}
    for season in seasons:
        csvs = list((Path(root) / season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df["season"] = season
        df["region"] = df["region"].str.strip().str.lower()
        df["s2_fileName"] = df["s2_fileName"].str.replace("\\", "/", regex=False)
        df["s2"] = root_str + "/" + df["s2_fileName"]
        buckets[season] = df[["s2", "season", "region"]].to_dict("records")
        print(f"  [{season}] {len(buckets[season]):,} optical images")
    pairs = [p for s in seasons if buckets[s] for p in buckets[s]]
    random.shuffle(pairs)
    return pairs


class OpticalDataset(Dataset):
    def __init__(self, pairs, img_size=512, augment=True):
        self.pairs = pairs; self.img_size = img_size; self.augment = augment

    def __len__(self): return len(self.pairs)

    def _load(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1) * 2.0 - 1.0

    def __getitem__(self, idx):
        opt = self._load(self.pairs[idx]["s2"])
        if self.augment:
            if random.random() > 0.5: opt = TF.hflip(opt)
            if random.random() > 0.5: opt = TF.vflip(opt)
            k = random.randint(0, 3)
            if k: opt = torch.rot90(opt, k, [1, 2])
        return {"optical": opt}


def build_fixed_val(val_pairs, n=2):
    rng = random.Random(FIXED_VAL_SEED)
    sel = []
    for s in SEASONS:
        pool = [p for p in val_pairs if p["season"] == s]
        if pool: sel.extend(rng.sample(pool, min(n, len(pool))))
    return sel


# ── Detail encoder: image(3) -> z_d (12ch), 8x downsample to match VAE ────────
class DetailEncoder(nn.Module):
    def __init__(self, in_ch=3, out_ch=DETAIL_CHANNELS, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, nf, 3, 2, 1),       # 512 -> 256
            nn.GroupNorm(8, nf), nn.SiLU(),
            nn.Conv2d(nf, nf * 2, 3, 2, 1),      # 256 -> 128
            nn.GroupNorm(8, nf * 2), nn.SiLU(),
            nn.Conv2d(nf * 2, nf * 4, 3, 2, 1),  # 128 -> 64
            nn.GroupNorm(8, nf * 4), nn.SiLU(),
            nn.Conv2d(nf * 4, nf * 4, 3, 1, 1),
            nn.GroupNorm(8, nf * 4), nn.SiLU(),
            nn.Conv2d(nf * 4, out_ch, 3, 1, 1),  # -> 12ch at 64x64
        )

    def forward(self, x): return self.net(x)


# ── Detail-Aligned VAE wrapper ────────────────────────────────────────────────
class DetailAlignedVAE(nn.Module):
    def __init__(self, base_vae):
        super().__init__()
        self.base_vae   = base_vae          # SD-1.5 VAE (encoder frozen)
        self.detail_enc = DetailEncoder()

        # Widen decoder.conv_in: Conv2d(4,512) -> Conv2d(16,512), zero-init extra
        old = base_vae.decoder.conv_in
        new = nn.Conv2d(TOTAL_CHANNELS, old.out_channels,
                        old.kernel_size, old.stride, old.padding,
                        bias=(old.bias is not None))
        with torch.no_grad():
            new.weight.zero_()
            new.weight[:, :BASE_CHANNELS].copy_(old.weight)  # base path preserved
            # detail input weights stay ZERO -> at init, decoder == vanilla
            if old.bias is not None:
                new.bias.copy_(old.bias)
        base_vae.decoder.conv_in = new

        # Freeze encoder + quant_conv (base latent must stay stable for the UNet)
        for p in base_vae.encoder.parameters():    p.requires_grad_(False)
        for p in base_vae.quant_conv.parameters(): p.requires_grad_(False)
        # Trainable: detail_enc + decoder (incl. widened conv_in) + post_quant_conv

    @torch.no_grad()
    def encode_base(self, x):
        return self.base_vae.encode(x).latent_dist.mode()   # (B,4,64,64) deterministic

    def encode_detail(self, x):
        return self.detail_enc(x)                            # (B,12,64,64)

    def decode_full(self, z_base, z_detail):
        z_base_pq = self.base_vae.post_quant_conv(z_base)    # (B,4,64,64)
        combined  = torch.cat([z_base_pq, z_detail], dim=1)  # (B,16,64,64)
        return self.base_vae.decoder(combined)               # (B,3,512,512)

    def forward(self, x):
        z_base   = self.encode_base(x)
        z_detail = self.encode_detail(x)
        recon    = self.decode_full(z_base, z_detail)
        return recon, z_base, z_detail


# ── Alignment loss: project z_d to 4ch via grouped averaging, L2 to base ──────
def alignment_loss(z_base, z_detail):
    B, D, H, W = z_detail.shape
    r = D // BASE_CHANNELS                       # 12 // 4 = 3
    grouped = z_detail.view(B, BASE_CHANNELS, r, H, W).mean(dim=2)  # (B,4,64,64)
    return F.mse_loss(grouped, z_base)


# ── Ceiling test: vanilla 4ch VAE vs detail-aligned 16ch, on same images ──────
@torch.no_grad()
def measure_ceiling(model, vanilla_vae, val_pairs, lpips_fn, n=N_CEILING_EVAL):
    model.eval()
    subset = random.Random(123).sample(val_pairs, min(n, len(val_pairs)))
    v_psnr, v_lpips, d_psnr, d_lpips = [], [], [], []
    for pair in subset:
        img = OpticalDataset([pair], IMG_SIZE, augment=False)[0]["optical"]
        x   = img.unsqueeze(0).to(device)

        # vanilla SD-1.5 VAE reconstruction (4ch)
        zb_v   = vanilla_vae.encode(x).latent_dist.mode()
        rec_v  = vanilla_vae.decode(zb_v).sample.clamp(-1, 1)

        # detail-aligned reconstruction (16ch)
        rec_d, _, _ = model(x)
        rec_d = rec_d.clamp(-1, 1)

        mse_v = F.mse_loss(rec_v, x).item()
        mse_d = F.mse_loss(rec_d, x).item()
        v_psnr.append(10 * np.log10(4.0 / mse_v))   # data range 2 -> max^2=4
        d_psnr.append(10 * np.log10(4.0 / mse_d))
        v_lpips.append(lpips_fn(rec_v, x).item())
        d_lpips.append(lpips_fn(rec_d, x).item())
    model.train()
    return (np.mean(v_psnr), np.mean(v_lpips),
            np.mean(d_psnr), np.mean(d_lpips))


def denorm(t): return ((t.clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy()


def show_recon(model, vanilla_vae, fixed_val, step):
    model.eval()
    n = len(fixed_val)
    fig, axes = plt.subplots(n, 3, figsize=(12, 3.5 * n))
    if n == 1: axes = axes.reshape(1, -1)
    with torch.no_grad():
        for i, pair in enumerate(fixed_val):
            img = OpticalDataset([pair], IMG_SIZE, augment=False)[0]["optical"]
            x   = img.unsqueeze(0).to(device)
            rec_v = vanilla_vae.decode(vanilla_vae.encode(x).latent_dist.mode()).sample
            rec_d, _, _ = model(x)
            axes[i, 0].imshow(denorm(img));            axes[i, 0].set_title("GT optical", fontsize=8); axes[i, 0].axis("off")
            axes[i, 1].imshow(denorm(rec_v[0]));       axes[i, 1].set_title("Vanilla 4ch", fontsize=8); axes[i, 1].axis("off")
            axes[i, 2].imshow(denorm(rec_d[0]));       axes[i, 2].set_title(f"Detail 16ch (step {step})", fontsize=8); axes[i, 2].axis("off")
    plt.suptitle(f"Reconstruction — step {step}", fontsize=11, fontweight="bold")
    plt.tight_layout(); display(fig); plt.close(fig)
    model.train()


def push_to_hf(folder, tag):
    if not HF_WRITE_TOKEN: return
    try:
        HfApi().upload_folder(folder_path=folder, path_in_repo=tag,
                              repo_id=HF_REPO_ID, repo_type="model", token=HF_WRITE_TOKEN)
        print(f"  HF push -> {tag}")
    except Exception as e:
        print(f"  HF push failed: {e}")


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    accelerator = Accelerator(mixed_precision="no",  # fp32 for precise ceiling measurement
                              gradient_accumulation_steps=GRAD_ACCUM,
                              project_dir=OUTPUT_DIR)
    set_seed(SEED)
    is_main = accelerator.is_main_process
    if is_main:
        os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("Loading SD-1.5 VAE (base, frozen encoder)...")
    base_vae    = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    # A second, untouched copy for the vanilla baseline in the ceiling test
    vanilla_vae = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae").to(device).eval()
    vanilla_vae.requires_grad_(False)

    model = DetailAlignedVAE(base_vae)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    print(f"  Detail-Aligned VAE ready. Trainable: {n_train:.1f}M params "
          f"(detail encoder + decoder). Base encoder frozen.")

    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)

    # Data
    all_pairs = collect_pairs(Path(DATA_ROOT), SEASONS)
    train_pairs, val_pairs = train_test_split(
        all_pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['region']}" for p in all_pairs],
        random_state=SEED)
    fixed_val = build_fixed_val(val_pairs)
    print(f"Train: {len(train_pairs):,} | Val: {len(val_pairs):,}")

    train_ds = OpticalDataset(train_pairs, IMG_SIZE, augment=True)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True,
                          persistent_workers=True, prefetch_factor=2)

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=LR, betas=(0.9, 0.999), weight_decay=1e-4)
    from diffusers.optimization import get_cosine_schedule_with_warmup
    lr_sched = get_cosine_schedule_with_warmup(optimizer, LR_WARMUP, NUM_TRAIN_STEPS)

    model, optimizer, train_dl, lr_sched = accelerator.prepare(
        model, optimizer, train_dl, lr_sched)

    # Baseline ceiling BEFORE any training (sanity: detail VAE should ~= vanilla at step 0)
    if is_main:
        vp, vl, dp, dl = measure_ceiling(accelerator.unwrap_model(model),
                                         vanilla_vae, val_pairs, lpips_fn, n=100)
        print("\n" + "="*58)
        print("CEILING @ step 0 (zero-init: detail VAE should match vanilla)")
        print(f"  Vanilla 4ch : PSNR {vp:.3f} dB | LPIPS {vl:.4f}")
        print(f"  Detail 16ch : PSNR {dp:.3f} dB | LPIPS {dl:.4f}")
        print("="*58 + "\n")

    global_step = 0
    ema_l1 = ema_lp = ema_al = 0.0; alpha = 0.98
    steps_per_epoch = len(train_dl) // GRAD_ACCUM
    total_epochs    = (NUM_TRAIN_STEPS + steps_per_epoch - 1) // steps_per_epoch

    bar = tqdm(total=NUM_TRAIN_STEPS, desc="Stage 1: Detail-Aligned VAE",
               disable=not is_main, dynamic_ncols=True)
    model.train()

    for epoch in range(total_epochs):
        for batch in train_dl:
            with accelerator.accumulate(model):
                x = batch["optical"].to(device)
                recon, z_base, z_detail = model(x)

                loss_l1  = F.l1_loss(recon, x) * L1_WEIGHT
                loss_lp  = lpips_fn(recon.clamp(-1, 1), x).mean() * LPIPS_WEIGHT
                loss_al  = alignment_loss(z_base, z_detail) * ALIGN_WEIGHT
                loss     = loss_l1 + loss_lp + loss_al

                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(trainable, 1.0)
                optimizer.step(); lr_sched.step(); optimizer.zero_grad(set_to_none=True)

            if accelerator.sync_gradients:
                global_step += 1
                ema_l1 = alpha*ema_l1 + (1-alpha)*loss_l1.item()
                ema_lp = alpha*ema_lp + (1-alpha)*loss_lp.item()
                ema_al = alpha*ema_al + (1-alpha)*loss_al.item()
                if is_main:
                    bar.update(1)
                    bar.set_postfix({"l1": f"{ema_l1:.4f}", "lpips": f"{ema_lp:.4f}",
                                     "align": f"{ema_al:.4f}"})

                if global_step % CEILING_EVAL_EVERY == 0 and is_main:
                    vp, vl, dp, dl = measure_ceiling(accelerator.unwrap_model(model),
                                                     vanilla_vae, val_pairs, lpips_fn)
                    tqdm.write("\n" + "-"*58)
                    tqdm.write(f"CEILING @ step {global_step}")
                    tqdm.write(f"  Vanilla 4ch : PSNR {vp:.3f} dB | LPIPS {vl:.4f}")
                    tqdm.write(f"  Detail 16ch : PSNR {dp:.3f} dB | LPIPS {dl:.4f}")
                    tqdm.write(f"  >>> GAIN    : PSNR {dp-vp:+.3f} dB | LPIPS {dl-vl:+.4f}")
                    tqdm.write("-"*58 + "\n")

                if global_step % VIS_EVERY == 0 and is_main:
                    show_recon(accelerator.unwrap_model(model), vanilla_vae, fixed_val, global_step)

                if global_step % SAVE_EVERY == 0 and is_main:
                    ckpt = os.path.join(OUTPUT_DIR, f"step_{global_step}")
                    os.makedirs(ckpt, exist_ok=True)
                    m = accelerator.unwrap_model(model)
                    torch.save(m.detail_enc.state_dict(), os.path.join(ckpt, "detail_encoder.pth"))
                    torch.save(m.base_vae.decoder.state_dict(), os.path.join(ckpt, "decoder.pth"))
                    torch.save({"base_channels": BASE_CHANNELS,
                                "detail_channels": DETAIL_CHANNELS,
                                "total_channels": TOTAL_CHANNELS},
                               os.path.join(ckpt, "detail_vae_config.pt"))
                    latest = os.path.join(OUTPUT_DIR, "latest")
                    if os.path.exists(latest): shutil.rmtree(latest)
                    shutil.copytree(ckpt, latest)
                    tqdm.write(f"  saved step {global_step}")
                    push_to_hf(ckpt, f"detail_vae_step_{global_step}")

            if global_step >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if global_step >= NUM_TRAIN_STEPS: break

    bar.close()
    if is_main:
        vp, vl, dp, dl = measure_ceiling(accelerator.unwrap_model(model),
                                         vanilla_vae, val_pairs, lpips_fn, n=300)
        print("\n" + "="*58)
        print("FINAL CEILING TEST (300 patches)")
        print(f"  Vanilla SD-1.5 4ch : PSNR {vp:.3f} dB | LPIPS {vl:.4f}")
        print(f"  Detail-Aligned 16ch: PSNR {dp:.3f} dB | LPIPS {dl:.4f}")
        print(f"  CEILING LIFT       : PSNR {dp-vp:+.3f} dB | LPIPS {dl-vl:+.4f}")
        print("="*58)
        if dp - vp >= 2.0:
            print("  VERDICT: Ceiling lifted >=2 dB. Stage 2 (bridge on 16ch) is worth it.")
        elif dp - vp >= 0.5:
            print("  VERDICT: Modest lift. Stage 2 may help; consider more detail channels.")
        else:
            print("  VERDICT: No meaningful lift. STOP — do not spend on Stage 2. Rethink.")
        final = os.path.join(OUTPUT_DIR, "final")
        os.makedirs(final, exist_ok=True)
        m = accelerator.unwrap_model(model)
        torch.save(m.detail_enc.state_dict(), os.path.join(final, "detail_encoder.pth"))
        torch.save(m.base_vae.decoder.state_dict(), os.path.join(final, "decoder.pth"))
        torch.save({"base_channels": BASE_CHANNELS, "detail_channels": DETAIL_CHANNELS,
                    "total_channels": TOTAL_CHANNELS}, os.path.join(final, "detail_vae_config.pt"))
        push_to_hf(final, "detail_vae_final")
    accelerator.end_training()


if __name__ == "__main__":
    main()

Writing /content/stage1_detail_vae.py


In [ ]:
import os
os.environ["HF_WRITE_TOKEN"] = "REDACTED_HF_TOKEN"
!pip install -q lpips
!python /content/stage1_detail_vae.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading SD-1.5 VAE (base, frozen encoder)...
  Detail-Aligned VAE ready. Trainable: 50.5M params (detail encoder + decoder). Base encoder frozen.
Setting up [LPIPS] perceptual loss: trunk

In [ ]:
%%writefile /content/check_dataset_structure.py
"""
Diagnostic: does the same physical location appear across seasons?
If yes -> a temporal/cross-season signal exists (changes the modeling options).
If no  -> seasons are just per-patch attribute labels (standard SEN1-2).

Checks three independent ways and prints evidence for each:
  (A) geographic coordinate columns (gold standard, if present)
  (B) location IDs parsed from s2 filenames with the season token removed
  (C) raw cross-season filename / ROI overlap
"""
import re
import pandas as pd
from pathlib import Path
from collections import defaultdict

DATA_ROOT = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
SEASONS   = ("spring", "summer", "fall", "winter")
pd.set_option("display.max_columns", 50, "display.width", 200)


def load_season_csvs(root, seasons):
    dfs = {}
    for s in seasons:
        folder = Path(root) / s
        csvs = list(folder.glob("*.csv"))
        if not csvs:
            print(f"  [{s}] no CSV found in {folder}")
            continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        dfs[s] = df
        print(f"  [{s}] {len(df):,} rows from {len(csvs)} csv(s)")
    return dfs


def section(t): print("\n" + "=" * 70 + f"\n{t}\n" + "=" * 70)


def main():
    section("0. DISCOVER + LOAD CSVs")
    dfs = load_season_csvs(DATA_ROOT, SEASONS)
    if not dfs:
        print("No CSVs loaded — check DATA_ROOT path.")
        return
    any_df = next(iter(dfs.values()))

    section("1. CSV STRUCTURE (columns + 3 sample rows from first season)")
    print("Columns:", list(any_df.columns))
    print()
    print(any_df.head(3).to_string())

    # ---------------------------------------------------------------- (A) coords
    section("A. COORDINATE-COLUMN CHECK (gold standard for 'same location')")
    coord_candidates = [c for c in any_df.columns
                        if re.search(r"(lat|lon|lng|x|y|coord|geo|east|north)", c, re.I)]
    if coord_candidates:
        print("Possible coordinate columns:", coord_candidates)
        # build a rounded-coordinate key per season and check overlap
        # try common lat/lon pairings
        lat_col = next((c for c in coord_candidates if re.search(r"lat", c, re.I)), None)
        lon_col = next((c for c in coord_candidates if re.search(r"lon|lng", c, re.I)), None)
        if lat_col and lon_col:
            print(f"Using lat='{lat_col}', lon='{lon_col}' (rounded to 3 decimals ~100m)")
            season_coords = {}
            for s, df in dfs.items():
                if lat_col in df and lon_col in df:
                    keys = set(zip(df[lat_col].round(3), df[lon_col].round(3)))
                    season_coords[s] = keys
                    print(f"  [{s}] {len(keys):,} unique coordinate cells")
            seasons_with = list(season_coords)
            shared = None
            for s in seasons_with:
                shared = season_coords[s] if shared is None else (shared & season_coords[s])
            if shared is not None:
                print(f"\n  >>> coordinate cells present in ALL {len(seasons_with)} seasons: {len(shared):,}")
                # pairwise too
                for i in range(len(seasons_with)):
                    for j in range(i + 1, len(seasons_with)):
                        a, b = seasons_with[i], seasons_with[j]
                        ov = len(season_coords[a] & season_coords[b])
                        print(f"      {a} ∩ {b}: {ov:,} shared cells")
        else:
            print("Found coordinate-like columns but no clear lat/lon pair; inspect manually above.")
    else:
        print("No coordinate columns detected. Falling back to filename-based checks.")

    # ---------------------------------------------------------------- (B) IDs
    section("B. LOCATION-ID FROM FILENAME (season token removed)")
    fname_col = next((c for c in any_df.columns
                     if re.search(r"s2.*file|file.*s2|s2_?name|optical", c, re.I)), None)
    if fname_col is None:
        fname_col = next((c for c in any_df.columns if re.search(r"file|name|path", c, re.I)), None)
    print(f"Using filename column: '{fname_col}'")
    if fname_col:
        print("Sample raw filenames:")
        for v in any_df[fname_col].head(5):
            print("   ", v)

        def loc_key(fn):
            base = Path(str(fn).replace("\\", "/")).name.lower()
            # remove season words, s1/s2 tokens, extension -> keep ROI/scene/patch
            base = re.sub(r"\.(png|jpg|jpeg|tif|tiff)$", "", base)
            for s in SEASONS:
                base = base.replace(s, "")
            base = re.sub(r"s[12]", "", base)
            base = re.sub(r"[_]+", "_", base).strip("_")
            return base

        print("\nSample location keys (season stripped):")
        for v in any_df[fname_col].head(5):
            print(f"    {v}  ->  {loc_key(v)}")

        season_keys = {}
        for s, df in dfs.items():
            if fname_col in df:
                season_keys[s] = set(df[fname_col].map(loc_key))
                print(f"  [{s}] {len(season_keys[s]):,} unique location keys")
        ss = list(season_keys)
        shared = None
        for s in ss:
            shared = season_keys[s] if shared is None else (shared & season_keys[s])
        if shared is not None:
            print(f"\n  >>> location keys present in ALL {len(ss)} seasons: {len(shared):,}")
            for i in range(len(ss)):
                for j in range(i + 1, len(ss)):
                    a, b = ss[i], ss[j]
                    print(f"      {a} ∩ {b}: {len(season_keys[a] & season_keys[b]):,}")
            if shared:
                print("  Example shared keys:", list(shared)[:5])

    # ---------------------------------------------------------------- (C) ROI
    section("C. ROI / SCENE-NUMBER OVERLAP (coarser than patch-level)")
    if fname_col:
        def roi_key(fn):
            base = Path(str(fn).replace("\\", "/")).name.lower()
            m = re.search(r"roi[s]?\d+", base)
            return m.group(0) if m else None
        season_rois = {}
        for s, df in dfs.items():
            if fname_col in df:
                rois = set(r for r in df[fname_col].map(roi_key) if r)
                season_rois[s] = rois
                print(f"  [{s}] {len(rois):,} unique ROI/scene ids "
                      f"(sample: {list(rois)[:3]})")
        ss = list(season_rois)
        if ss and all(season_rois[s] for s in ss):
            for i in range(len(ss)):
                for j in range(i + 1, len(ss)):
                    a, b = ss[i], ss[j]
                    print(f"      {a} ∩ {b}: {len(season_rois[a] & season_rois[b]):,} shared ROIs")

    section("VERDICT GUIDE")
    print("""If A/B/C all show ~0 cross-season overlap -> seasons are independent
patches (standard SEN1-2). No temporal graph available; proceed VAE+bridge,
no GNN. If overlap is substantial -> the SAME locations recur across seasons,
which WOULD justify a cross-season / temporal modeling component. Read the
three sections above and tell me the overlap numbers.""")


if __name__ == "__main__":
    main()

Writing /content/check_dataset_structure.py


In [ ]:
!python /content/check_dataset_structure.py


0. DISCOVER + LOAD CSVs
  [spring] 72,019 rows from 69 csv(s)
  [summer] 59,764 rows from 49 csv(s)
  [fall] 94,202 rows from 74 csv(s)
  [winter] 81,895 rows from 64 csv(s)

1. CSV STRUCTURE (columns + 3 sample rows from first season)
Columns: ['s1_fileName', 's2_fileName', 'season', 'region']

                                   s1_fileName                                  s2_fileName  season     region
0    spring/s1_65/ROIs1158_spring1s2_65_p1.png    spring/s2_65/ROIs1158_spring_s2_65_p1.png  spring  Temperate
1   spring/s1_65/ROIs1158_spring1s2_65_p10.png   spring/s2_65/ROIs1158_spring_s2_65_p10.png  spring  Temperate
2  spring/s1_65/ROIs1158_spring1s2_65_p100.png  spring/s2_65/ROIs1158_spring_s2_65_p100.png  spring  Temperate

A. COORDINATE-COLUMN CHECK (gold standard for 'same location')
No coordinate columns detected. Falling back to filename-based checks.

B. LOCATION-ID FROM FILENAME (season token removed)
Using filename column: 's2_fileName'
Sample raw filenames:
    spring/

In [ ]:
%%writefile /content/stage2_adapted16_256.py
"""
SAR-INTEL Stage 2 (ADAPTED) — fully end-to-end 16-channel SD-1.5 VAE @ 256px.

Removes the frozen-base weakness: the ENTIRE SD-1.5 autoencoder is unfrozen and
co-adapted to Sentinel-2 optical as a single native 16-channel latent.
Initialized from the Stage-1 detail VAE (37.7 dB) so we keep the head start.

Recipe (verified from LDM / Diffusability / practitioner sources):
  L = L1 + LPIPS + kl*KL(annealed 1e-6) + 0.5*align + 0.25*SE + adv*hinge(D)
  - KL very low (1e-6) annealed  -> prevents posterior collapse
  - PatchGAN adversarial, delayed start, hinge, D updated on detached recon
  - scale-equivariance (0.25)    -> keeps the 16ch latent diffusable
  - DA-VAE alignment (0.5)        -> keeps 16 channels structured (base+detail)
256px so results are comparable to published SAR->optical benchmarks.
"""
import os, gc, random, warnings, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from diffusers import AutoencoderKL
from diffusers.optimization import get_cosine_schedule_with_warmup
from accelerate import Accelerator
from accelerate.utils import set_seed
from tqdm.auto import tqdm
from huggingface_hub import HfApi, hf_hub_download
import lpips as lpips_lib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.fftpack import dctn
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ── Config ────────────────────────────────────────────────────────────────────
DATA_ROOT       = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR      = "/content/drive/MyDrive/stage2_adapted16"
SD_MODEL_ID     = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID      = "AliMusaRizvi/sar-to-optical-diffusion"
HF_WRITE_TOKEN  = os.environ.get("HF_WRITE_TOKEN", "")
RESUME_TAG      = "detail_vae_final"   # Stage 1 output

IMG_SIZE        = 256
SEASONS         = ("spring", "summer", "fall", "winter")
VAL_FRACTION    = 0.05
SEED            = 42

BASE_CHANNELS   = 4
DETAIL_CHANNELS = 12
TOTAL_CHANNELS  = BASE_CHANNELS + DETAIL_CHANNELS

NUM_TRAIN_STEPS = 20_000
BATCH_SIZE      = 8
LR_AE           = 1e-5      # low: protect the 37.7 dB init, stable adaptation
LR_DISC         = 1e-5
LR_WARMUP       = 300

# loss weights
L1_WEIGHT       = 1.0
LPIPS_WEIGHT    = 1.0
ALIGN_WEIGHT    = 0.5
SE_WEIGHT       = 0.25
ADV_WEIGHT      = 0.1       # gentle (recipe: keep adversarial small)
KL_WEIGHT_MAX   = 1e-6      # very low (prevents posterior collapse)
KL_ANNEAL_STEPS = 2_000
DISC_START      = 1_000     # let reconstruction settle before adversarial
SE_FACTORS      = (2, 4)

CEILING_EVAL_EVERY = 3_000
N_CEILING_EVAL     = 200
DCT_SAMPLES        = 32
SAVE_EVERY         = 4_000
FIXED_VAL_SEED     = 999

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ── Data ──────────────────────────────────────────────────────────────────────
def collect_pairs(root, seasons):
    root_str = Path(root).as_posix()
    buckets  = {s: [] for s in seasons}
    for season in seasons:
        csvs = list((Path(root) / season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df["season"] = season
        df["region"] = df["region"].str.strip().str.lower()
        df["s2_fileName"] = df["s2_fileName"].str.replace("\\", "/", regex=False)
        df["s2"] = root_str + "/" + df["s2_fileName"]
        buckets[season] = df[["s2", "season", "region"]].to_dict("records")
        print(f"  [{season}] {len(buckets[season]):,} optical")
    pairs = [p for s in seasons if buckets[s] for p in buckets[s]]
    random.shuffle(pairs)
    return pairs


class OpticalDataset(Dataset):
    def __init__(self, pairs, img_size=256, augment=True):
        self.pairs = pairs; self.img_size = img_size; self.augment = augment
    def __len__(self): return len(self.pairs)
    def _load(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1) * 2.0 - 1.0
    def __getitem__(self, idx):
        opt = self._load(self.pairs[idx]["s2"])
        if self.augment:
            if random.random() > 0.5: opt = TF.hflip(opt)
            if random.random() > 0.5: opt = TF.vflip(opt)
            k = random.randint(0, 3)
            if k: opt = torch.rot90(opt, k, [1, 2])
        return {"optical": opt}


def build_fixed_val(val_pairs, n=2):
    rng = random.Random(FIXED_VAL_SEED)
    sel = []
    for s in SEASONS:
        pool = [p for p in val_pairs if p["season"] == s]
        if pool: sel.extend(rng.sample(pool, min(n, len(pool))))
    return sel


# ── Detail encoder + adapted VAE wrapper (EVERYTHING trainable) ───────────────
class DetailEncoder(nn.Module):
    def __init__(self, in_ch=3, out_ch=DETAIL_CHANNELS, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, nf, 3, 2, 1),       nn.GroupNorm(8, nf), nn.SiLU(),
            nn.Conv2d(nf, nf * 2, 3, 2, 1),      nn.GroupNorm(8, nf * 2), nn.SiLU(),
            nn.Conv2d(nf * 2, nf * 4, 3, 2, 1),  nn.GroupNorm(8, nf * 4), nn.SiLU(),
            nn.Conv2d(nf * 4, nf * 4, 3, 1, 1),  nn.GroupNorm(8, nf * 4), nn.SiLU(),
            nn.Conv2d(nf * 4, out_ch, 3, 1, 1),
        )
    def forward(self, x): return self.net(x)


class AdaptedVAE(nn.Module):
    """Native 16ch latent. base (variational, 4ch) + detail (deterministic, 12ch).
       ENTIRE module trainable — no frozen base."""
    def __init__(self, base_vae):
        super().__init__()
        self.base_vae   = base_vae
        self.detail_enc = DetailEncoder()
        old = base_vae.decoder.conv_in
        new = nn.Conv2d(TOTAL_CHANNELS, old.out_channels,
                        old.kernel_size, old.stride, old.padding,
                        bias=(old.bias is not None))
        with torch.no_grad():
            new.weight.zero_()
            new.weight[:, :BASE_CHANNELS].copy_(old.weight)
            if old.bias is not None: new.bias.copy_(old.bias)
        base_vae.decoder.conv_in = new
        # NOTHING frozen — full end-to-end adaptation.

    def encode(self, x):
        base_dist = self.base_vae.encode(x).latent_dist     # grad flows (unfrozen)
        z_base    = base_dist.sample()                       # 4ch, stochastic
        z_detail  = self.detail_enc(x)                       # 12ch, deterministic
        return z_base, z_detail, base_dist

    def decode(self, z_base, z_detail):
        z_base_pq = self.base_vae.post_quant_conv(z_base)
        combined  = torch.cat([z_base_pq, z_detail], dim=1)  # 16ch
        return self.base_vae.decoder(combined), combined

    @torch.no_grad()
    def reconstruct(self, x):
        base_dist = self.base_vae.encode(x).latent_dist
        z_base    = base_dist.mode()                         # deterministic for eval
        z_detail  = self.detail_enc(x)
        recon, combined = self.decode(z_base, z_detail)
        return recon, combined


# ── PatchGAN discriminator (taming-transformers style) ────────────────────────
class NLayerDiscriminator(nn.Module):
    def __init__(self, in_ch=3, nf=64, n_layers=3):
        super().__init__()
        layers = [nn.Conv2d(in_ch, nf, 4, 2, 1), nn.LeakyReLU(0.2, True)]
        mult = 1
        for n in range(1, n_layers):
            mult_prev, mult = mult, min(2 ** n, 8)
            layers += [nn.Conv2d(nf * mult_prev, nf * mult, 4, 2, 1, bias=False),
                       nn.BatchNorm2d(nf * mult), nn.LeakyReLU(0.2, True)]
        mult_prev, mult = mult, min(2 ** n_layers, 8)
        layers += [nn.Conv2d(nf * mult_prev, nf * mult, 4, 1, 1, bias=False),
                   nn.BatchNorm2d(nf * mult), nn.LeakyReLU(0.2, True),
                   nn.Conv2d(nf * mult, 1, 4, 1, 1)]
        self.main = nn.Sequential(*layers)
    def forward(self, x): return self.main(x)


# ── Losses ────────────────────────────────────────────────────────────────────
def alignment_loss(z_base, z_detail):
    B, D, H, W = z_detail.shape
    r = D // BASE_CHANNELS
    grouped = z_detail.view(B, BASE_CHANNELS, r, H, W).mean(dim=2)
    return F.mse_loss(grouped, z_base)


def scale_equivariance_loss(model, x, combined):
    factor = random.choice(SE_FACTORS)
    x_down = F.interpolate(x, scale_factor=1.0 / factor, mode="bilinear", align_corners=False)
    c_down = F.interpolate(combined, scale_factor=1.0 / factor, mode="bilinear", align_corners=False)
    recon_down = model.base_vae.decoder(c_down)
    return F.l1_loss(recon_down, x_down)


def hinge_d(real_logits, fake_logits):
    return 0.5 * (F.relu(1.0 - real_logits).mean() + F.relu(1.0 + fake_logits).mean())


# ── Diagnostics ───────────────────────────────────────────────────────────────
def freq_profile(latent_bchw):
    arr = latent_bchw.detach().float().cpu().numpy()
    B, C, H, W = arr.shape
    maxk = (H - 1) + (W - 1)
    acc = np.zeros(maxk + 1); cnt = np.zeros(maxk + 1)
    for b in range(B):
        for c in range(C):
            D = np.abs(dctn(arr[b, c], norm="ortho")); D = D / (D[0, 0] + 1e-8)
            for u in range(H):
                for v in range(W):
                    acc[u + v] += D[u, v]; cnt[u + v] += 1
    prof = acc / np.maximum(cnt, 1)
    return np.linspace(0, 1, len(prof)), prof


def denorm(t): return ((t.clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy()


@torch.no_grad()
def measure_ceiling(model, vanilla_vae, val_pairs, lpips_fn, n=N_CEILING_EVAL):
    model.eval()
    subset = random.Random(123).sample(val_pairs, min(n, len(val_pairs)))
    vp, vl, dp, dl = [], [], [], []
    for pair in subset:
        img = OpticalDataset([pair], IMG_SIZE, augment=False)[0]["optical"]
        x   = img.unsqueeze(0).to(device)
        rec_v = vanilla_vae.decode(vanilla_vae.encode(x).latent_dist.mode()).sample.clamp(-1, 1)
        rec_d, _ = model.reconstruct(x); rec_d = rec_d.clamp(-1, 1)
        vp.append(10 * np.log10(4.0 / F.mse_loss(rec_v, x).item()))
        dp.append(10 * np.log10(4.0 / F.mse_loss(rec_d, x).item()))
        vl.append(lpips_fn(rec_v, x).item()); dl.append(lpips_fn(rec_d, x).item())
    model.train()
    return np.mean(vp), np.mean(vl), np.mean(dp), np.mean(dl)


@torch.no_grad()
def collect_profiles(model, vanilla_vae, val_pairs, n=DCT_SAMPLES):
    model.eval()
    subset = random.Random(7).sample(val_pairs, min(n, len(val_pairs)))
    van, det, rgb = [], [], []
    for pair in subset:
        img = OpticalDataset([pair], IMG_SIZE, augment=False)[0]["optical"]
        x   = img.unsqueeze(0).to(device)
        van.append(vanilla_vae.encode(x).latent_dist.mode().cpu())
        _, combined = model.reconstruct(x); det.append(combined.cpu())
        rgb.append(F.interpolate(x, size=(32, 32), mode="bilinear", align_corners=False).cpu())
    model.train()
    fa, pv = freq_profile(torch.cat(van))
    _,  pd_ = freq_profile(torch.cat(det))
    _,  pr  = freq_profile(torch.cat(rgb))
    return fa, pv, pd_, pr


def push_to_hf(folder, tag):
    if not HF_WRITE_TOKEN: return
    try:
        HfApi().upload_folder(folder_path=folder, path_in_repo=tag,
                              repo_id=HF_REPO_ID, repo_type="model", token=HF_WRITE_TOKEN)
        print(f"  HF push -> {tag}")
    except Exception as e:
        print(f"  HF push failed: {e}")


def load_stage1(model):
    de = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{RESUME_TAG}/detail_encoder.pth")
    dc = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{RESUME_TAG}/decoder.pth")
    model.detail_enc.load_state_dict(torch.load(de, map_location="cpu"))
    model.base_vae.decoder.load_state_dict(torch.load(dc, map_location="cpu"))
    print("  Init from Stage 1: detail_encoder + decoder loaded; encoder = SD-1.5 default.")
    print("  All components now UNFROZEN for end-to-end adaptation.")


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    accelerator = Accelerator(mixed_precision="no", gradient_accumulation_steps=1,
                              project_dir=OUTPUT_DIR)
    set_seed(SEED)
    is_main = accelerator.is_main_process
    if is_main: os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("Loading SD-1.5 VAE + building adapted 16ch model...")
    base_vae    = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    vanilla_vae = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae").to(device).eval()
    vanilla_vae.requires_grad_(False)

    model = AdaptedVAE(base_vae)
    load_stage1(model)
    disc  = NLayerDiscriminator(in_ch=3)
    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)
    print(f"  VAE trainable: {sum(p.numel() for p in model.parameters())/1e6:.1f}M | "
          f"Disc: {sum(p.numel() for p in disc.parameters())/1e6:.1f}M")

    all_pairs = collect_pairs(Path(DATA_ROOT), SEASONS)
    train_pairs, val_pairs = train_test_split(
        all_pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['region']}" for p in all_pairs], random_state=SEED)
    fixed_val = build_fixed_val(val_pairs)
    print(f"Train: {len(train_pairs):,} | Val: {len(val_pairs):,} | @ {IMG_SIZE}px")

    train_ds = OpticalDataset(train_pairs, IMG_SIZE, augment=True)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4,
                          pin_memory=True, drop_last=True, persistent_workers=True, prefetch_factor=2)

    opt_ae   = torch.optim.AdamW(model.parameters(), lr=LR_AE, betas=(0.5, 0.9), weight_decay=1e-4)
    opt_disc = torch.optim.AdamW(disc.parameters(),  lr=LR_DISC, betas=(0.5, 0.9))
    lr_sched = get_cosine_schedule_with_warmup(opt_ae, LR_WARMUP, NUM_TRAIN_STEPS)

    model, disc, opt_ae, opt_disc, train_dl, lr_sched = accelerator.prepare(
        model, disc, opt_ae, opt_disc, train_dl, lr_sched)

    # BEFORE snapshot
    if is_main:
        vp, vl, dp, dl = measure_ceiling(accelerator.unwrap_model(model), vanilla_vae, val_pairs, lpips_fn, 150)
        print("\n" + "="*62)
        print("BEFORE (256px, Stage-1 init, nothing adapted yet)")
        print(f"  Vanilla 4ch  : PSNR {vp:.3f} dB | LPIPS {vl:.4f}")
        print(f"  Adapted 16ch : PSNR {dp:.3f} dB | LPIPS {dl:.4f}  (gain {dp-vp:+.3f})")
        print("="*62 + "\n")
        fa, pv, pd_before, pr = collect_profiles(accelerator.unwrap_model(model), vanilla_vae, val_pairs)

    global_step = 0
    e_l1 = e_lp = e_kl = e_al = e_se = e_g = e_d = 0.0; a = 0.98
    steps_per_epoch = len(train_dl)
    total_epochs    = (NUM_TRAIN_STEPS + steps_per_epoch - 1) // steps_per_epoch
    bar = tqdm(total=NUM_TRAIN_STEPS, desc="Stage 2 adapted-16ch", disable=not is_main, dynamic_ncols=True)
    model.train(); disc.train()

    for epoch in range(total_epochs):
        for batch in train_dl:
            x = batch["optical"].to(device)
            z_base, z_detail, base_dist = model.encode(x)
            recon, combined = model.decode(z_base, z_detail)

            # reconstruction + regularizers
            l1   = F.l1_loss(recon, x) * L1_WEIGHT
            lp   = lpips_fn(recon.clamp(-1, 1), x).mean() * LPIPS_WEIGHT
            kl_w = KL_WEIGHT_MAX * min(1.0, global_step / KL_ANNEAL_STEPS)
            kl   = base_dist.kl().mean() * kl_w
            al   = alignment_loss(z_base, z_detail) * ALIGN_WEIGHT
            se   = scale_equivariance_loss(accelerator.unwrap_model(model), x, combined) * SE_WEIGHT
            adv_on = global_step >= DISC_START
            g_adv  = (-disc(recon).mean() * ADV_WEIGHT) if adv_on else torch.tensor(0.0, device=device)
            ae_loss = l1 + lp + kl + al + se + g_adv

            opt_ae.zero_grad(set_to_none=True)
            if adv_on: opt_disc.zero_grad(set_to_none=True)
            accelerator.backward(ae_loss)
            opt_ae.step(); lr_sched.step()

            # discriminator step
            d_loss_val = 0.0
            if adv_on:
                opt_disc.zero_grad(set_to_none=True)
                d_real = disc(x); d_fake = disc(recon.detach())
                d_loss = hinge_d(d_real, d_fake)
                accelerator.backward(d_loss); opt_disc.step()
                d_loss_val = d_loss.item()

            global_step += 1
            e_l1 = a*e_l1+(1-a)*l1.item(); e_lp = a*e_lp+(1-a)*lp.item()
            e_kl = a*e_kl+(1-a)*kl.item(); e_al = a*e_al+(1-a)*al.item()
            e_se = a*e_se+(1-a)*se.item(); e_g = a*e_g+(1-a)*float(g_adv); e_d = a*e_d+(1-a)*d_loss_val
            if is_main:
                bar.update(1)
                bar.set_postfix({"l1": f"{e_l1:.3f}", "lpips": f"{e_lp:.3f}", "se": f"{e_se:.3f}",
                                 "G": f"{e_g:.3f}", "D": f"{e_d:.3f}"})

            if global_step % CEILING_EVAL_EVERY == 0 and is_main:
                vp, vl, dp, dl = measure_ceiling(accelerator.unwrap_model(model), vanilla_vae, val_pairs, lpips_fn)
                tqdm.write(f"\n[step {global_step}] Adapted 16ch: PSNR {dp:.3f} | LPIPS {dl:.4f} "
                           f"(vanilla {vp:.3f}, gain {dp-vp:+.3f})\n")

            if global_step % SAVE_EVERY == 0 and is_main:
                ckpt = os.path.join(OUTPUT_DIR, f"step_{global_step}")
                os.makedirs(ckpt, exist_ok=True)
                m = accelerator.unwrap_model(model)
                torch.save(m.detail_enc.state_dict(), os.path.join(ckpt, "detail_encoder.pth"))
                torch.save(m.base_vae.state_dict(), os.path.join(ckpt, "base_vae.pth"))
                torch.save({"base_channels": BASE_CHANNELS, "detail_channels": DETAIL_CHANNELS,
                            "total_channels": TOTAL_CHANNELS, "img_size": IMG_SIZE,
                            "adapted": True}, os.path.join(ckpt, "adapted_vae_config.pt"))
                latest = os.path.join(OUTPUT_DIR, "latest")
                if os.path.exists(latest): shutil.rmtree(latest)
                shutil.copytree(ckpt, latest)
                push_to_hf(ckpt, f"adapted16_step_{global_step}")

            if global_step >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if global_step >= NUM_TRAIN_STEPS: break
    bar.close()

    if is_main:
        vp, vl, dp, dl = measure_ceiling(accelerator.unwrap_model(model), vanilla_vae, val_pairs, lpips_fn, 300)
        print("\n" + "="*62)
        print("AFTER  (256px, fully adapted 16ch + SE + adversarial)")
        print(f"  Vanilla 4ch  : PSNR {vp:.3f} dB | LPIPS {vl:.4f}")
        print(f"  Adapted 16ch : PSNR {dp:.3f} dB | LPIPS {dl:.4f}  (gain {dp-vp:+.3f})")
        print("="*62)
        _, _, pd_after, _ = collect_profiles(accelerator.unwrap_model(model), vanilla_vae, val_pairs)

        plt.figure(figsize=(8, 5))
        plt.semilogy(fa, pv,        label="SD vanilla 4ch", lw=2)
        plt.semilogy(fa, pd_before, label="Adapted 16ch (before)", lw=2, ls="--")
        plt.semilogy(fa, pd_after,  label="Adapted 16ch (after)", lw=2)
        plt.semilogy(fa, pr,        label="RGB reference", lw=2, color="black", alpha=0.6)
        plt.xlabel("Normalized frequency (low → high)"); plt.ylabel("Norm. DCT amplitude (log)")
        plt.title("Latent diffusability after end-to-end adaptation + SE")
        plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
        fp = os.path.join(OUTPUT_DIR, "diffusability_dct.png")
        plt.savefig(fp, dpi=150, bbox_inches="tight"); display(plt.gcf()); plt.close()

        final = os.path.join(OUTPUT_DIR, "final")
        os.makedirs(final, exist_ok=True)
        m = accelerator.unwrap_model(model)
        torch.save(m.detail_enc.state_dict(), os.path.join(final, "detail_encoder.pth"))
        torch.save(m.base_vae.state_dict(), os.path.join(final, "base_vae.pth"))
        torch.save({"base_channels": BASE_CHANNELS, "detail_channels": DETAIL_CHANNELS,
                    "total_channels": TOTAL_CHANNELS, "img_size": IMG_SIZE, "adapted": True},
                   os.path.join(final, "adapted_vae_config.pt"))
        shutil.copy(fp, os.path.join(final, "diffusability_dct.png"))
        push_to_hf(final, "adapted16_final")
        print("\nStage 2 (adapted) complete. Native 16ch SD-1.5 latent @ 256px.")
    accelerator.end_training()


if __name__ == "__main__":
    main()

Writing /content/stage2_adapted16_256.py


In [ ]:
import os
os.environ["HF_WRITE_TOKEN"] = "REDACTED_HF_TOKEN"
!pip install -q lpips scipy
!python /content/stage2_adapted16_256.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading SD-1.5 VAE + building adapted 16ch model...
detail_vae_final/detail_encoder.pth: 100% 3.97M/3.97M [00:00<00:00, 5.36MB/s]
detail_vae_final/decoder.pth: 100% 198M/198M [00:01<00:00

# Stage 3

In [ ]:
%%writefile /content/stage3_bridge_256.py
"""
SAR-INTEL Stage 3 — ResShift bridge on the adapted 16ch latent @ 256px.
RESUMES from the 80k full-state on HF ('bridge_resume'). No periodic HF pushes
(removed to kill upload overhead). Evaluates every 5k. Saves FINAL model ONCE
to HF in safetensors. A fast LOCAL-only checkpoint every 20k is the sole safety
net (disk write, no upload). Set LOCAL_SAVE_EVERY=0 to disable entirely.
"""
import os, gc, re, math, random, warnings, shutil, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed
from tqdm.auto import tqdm
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from safetensors.torch import save_file
import lpips as lpips_lib
import pandas as pd
from torchmetrics.image import StructuralSimilarityIndexMeasure
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ── Config ────────────────────────────────────────────────────────────────────
DATA_ROOT    = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
DRIVE_LATEST = "/content/drive/MyDrive/stage3_bridge/latest"  # legacy fallback only
OUTPUT_DIR   = "/content/stage3_bridge"
SD_MODEL_ID  = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID   = "AliMusaRizvi/sar-to-optical-diffusion"
HF_TOKEN     = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG      = "adapted16_final"
RESUME_TAG   = "bridge_resume"        # 80k full state lives here on HF (read once)

IMG_SIZE     = 256
SEASONS      = ("spring", "summer", "fall", "winter")
VAL_FRACTION = 0.05
SEED         = 42

BASE_CH, DETAIL_CH = 4, 12
TOTAL_CH     = 16

T_STEPS      = 15
KAPPA        = 2.0
ETA_1, ETA_T = 0.04, 0.99

NUM_TRAIN_STEPS = 150_000
BATCH_SIZE   = 4
LR           = 5e-5
LR_WARMUP    = 1_000
COND_DROPOUT = 0.10
VAL_EVERY    = 5_000
LOCAL_SAVE_EVERY = 20_000     # LOCAL disk only (fast, no upload). 0 = disable.
N_VAL        = 100
LOG_EVERY    = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEASON_DESC = {"spring": "spring", "summer": "summer", "fall": "autumn", "winter": "winter"}
TERRAINS = ("tropical", "temperate", "arctic", "arid", "coastal", "urban")

def map_region_to_terrain(region):
    r = (region or "").lower()
    if any(k in r for k in ("trop", "rainforest", "amazon", "jungle")): return "tropical"
    if any(k in r for k in ("arctic", "tundra", "snow", "ice", "polar")): return "arctic"
    if any(k in r for k in ("desert", "arid", "sahara", "dry")): return "arid"
    if any(k in r for k in ("coast", "beach", "shore", "marine", "sea")): return "coastal"
    if any(k in r for k in ("urban", "city", "metro")): return "urban"
    return "temperate"


# ── Data ──────────────────────────────────────────────────────────────────────
def collect_pairs(root, seasons):
    root_str = Path(root).as_posix()
    pairs = []
    for season in seasons:
        csvs = list((Path(root) / season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for col in ("s1_fileName", "s2_fileName"):
            df[col] = df[col].astype(str).str.replace("\\", "/", regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            pairs.append({"sar": f"{root_str}/{row['s1_fileName']}",
                          "opt": f"{root_str}/{row['s2_fileName']}",
                          "season": season,
                          "terrain": map_region_to_terrain(row["region"])})
    random.shuffle(pairs)
    return pairs


class PairDataset(Dataset):
    def __init__(self, pairs, img_size=256, augment=True):
        self.pairs = pairs; self.img_size = img_size; self.augment = augment
    def __len__(self): return len(self.pairs)
    def _load(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1) * 2.0 - 1.0
    def __getitem__(self, idx):
        p = self.pairs[idx]
        sar = self._load(p["sar"]); opt = self._load(p["opt"])
        if self.augment:
            if random.random() > 0.5: sar, opt = torch.flip(sar, [2]), torch.flip(opt, [2])
            if random.random() > 0.5: sar, opt = torch.flip(sar, [1]), torch.flip(opt, [1])
        return {"sar": sar, "opt": opt, "season": p["season"], "terrain": p["terrain"]}


# ── Adapted VAE (frozen) ──────────────────────────────────────────────────────
class DetailEncoder(nn.Module):
    def __init__(self, in_ch=3, out_ch=DETAIL_CH, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, nf, 3, 2, 1),      nn.GroupNorm(8, nf), nn.SiLU(),
            nn.Conv2d(nf, nf*2, 3, 2, 1),       nn.GroupNorm(8, nf*2), nn.SiLU(),
            nn.Conv2d(nf*2, nf*4, 3, 2, 1),     nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, nf*4, 3, 1, 1),     nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, out_ch, 3, 1, 1))
    def forward(self, x): return self.net(x)


class AdaptedVAE(nn.Module):
    def __init__(self, base_vae):
        super().__init__()
        self.base_vae = base_vae
        self.detail_enc = DetailEncoder()
        old = base_vae.decoder.conv_in
        base_vae.decoder.conv_in = nn.Conv2d(TOTAL_CH, old.out_channels, old.kernel_size,
                                             old.stride, old.padding, bias=(old.bias is not None))
    @torch.no_grad()
    def encode_latent(self, x):
        z_base = self.base_vae.encode(x).latent_dist.mode()
        z_base_pq = self.base_vae.post_quant_conv(z_base)
        z_detail = self.detail_enc(x)
        return torch.cat([z_base_pq, z_detail], dim=1)
    @torch.no_grad()
    def decode_latent(self, lat):
        return self.base_vae.decoder(lat)


def load_adapted_vae():
    base_vae = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    vae = AdaptedVAE(base_vae)
    bp = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/base_vae.pth")
    de = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/detail_encoder.pth")
    vae.base_vae.load_state_dict(torch.load(bp, map_location="cpu"))
    vae.detail_enc.load_state_dict(torch.load(de, map_location="cpu"))
    vae = vae.to(device).eval().float(); vae.requires_grad_(False)
    print(f"  Adapted VAE loaded from {VAE_TAG}.")
    return vae


def build_embed_cache():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    txt.requires_grad_(False)
    cache = {}
    @torch.no_grad()
    def embed(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s, ter)] = embed(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = embed("")
    del txt; gc.collect(); torch.cuda.empty_cache()
    print(f"  Built {len(cache)} CLIP embeddings.")
    return cache


def build_unet():
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    ni = nn.Conv2d(2 * TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    with torch.no_grad():
        ni.weight.zero_()
        for i in range(2 * TOTAL_CH): ni.weight[:, i] = oi.weight[:, i % oi.in_channels] / 8.0
        ni.bias.copy_(oi.bias)
    unet.conv_in = ni; unet.config.in_channels = 2 * TOTAL_CH
    oo = unet.conv_out
    no = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    with torch.no_grad():
        for i in range(TOTAL_CH):
            no.weight[i] = oo.weight[i % oo.out_channels]; no.bias[i] = oo.bias[i % oo.out_channels]
    unet.conv_out = no; unet.config.out_channels = TOTAL_CH
    print(f"  UNet: conv_in {2*TOTAL_CH}ch, conv_out {TOTAL_CH}ch.")
    return unet


def make_eta(T, e1, eT):
    idx = np.arange(T)
    return torch.tensor(e1 * (eT / e1) ** (idx / (T - 1)), dtype=torch.float32)


def bridge_forward(x0, y, t, eta, kappa):
    et = eta[t].view(-1, 1, 1, 1).to(x0.device)
    return x0 + et * (y - x0) + kappa * torch.sqrt(et) * torch.randn_like(x0)


@torch.no_grad()
def bridge_sample(unet, y, cond, eta, kappa, dtype):
    B = y.shape[0]; T = len(eta)
    x = y + kappa * torch.sqrt(eta[-1]).to(y.device) * torch.randn_like(y)
    for t in reversed(range(T)):
        tb = torch.full((B,), int(t * 1000 / T), device=y.device, dtype=torch.long)
        x0 = unet(torch.cat([x, y], 1).to(dtype), tb, encoder_hidden_states=cond.to(dtype)).sample.float()
        if t > 0:
            ep = eta[t - 1].to(y.device)
            x = x0 + ep * (y - x0) + kappa * torch.sqrt(ep) * torch.randn_like(x)
        else:
            x = x0
    return x


def get_cond(seasons, terrains, cache, drop=0.0):
    out = []
    for s, ter in zip(seasons, terrains):
        if drop > 0 and random.random() < drop: out.append(cache["null"])
        else: out.append(cache.get((s, ter), cache["null"]))
    return torch.stack(out)


def push_hf(folder, tag):
    if not HF_TOKEN: return
    try:
        HfApi().upload_folder(folder_path=folder, path_in_repo=tag,
                              repo_id=HF_REPO_ID, repo_type="model", token=HF_TOKEN)
        print(f"  HF push -> {tag}")
    except Exception as e:
        print(f"  HF push failed: {e}")


def download_resume_from_hf(dest):
    try:
        snapshot_download(repo_id=HF_REPO_ID, allow_patterns=f"{RESUME_TAG}/*",
                          local_dir="/content/_resume_dl", token=HF_TOKEN)
        src = os.path.join("/content/_resume_dl", RESUME_TAG)
        if os.path.exists(os.path.join(src, "meta.pt")):
            if os.path.exists(dest): shutil.rmtree(dest)
            shutil.copytree(src, dest); return True
    except Exception as e:
        print(f"  no HF resume state ({e})")
    return False


def main():
    accelerator = Accelerator(mixed_precision="bf16", project_dir=OUTPUT_DIR)
    set_seed(SEED)
    is_main = accelerator.is_main_process
    dtype = torch.bfloat16
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("Loading components...")
    vae = load_adapted_vae()
    embed_cache = build_embed_cache()
    unet = build_unet()
    eta = make_eta(T_STEPS, ETA_1, ETA_T).to(device)
    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)

    all_pairs = collect_pairs(Path(DATA_ROOT), SEASONS)
    train_pairs, val_pairs = train_test_split(
        all_pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in all_pairs], random_state=SEED)
    print(f"Train: {len(train_pairs):,} | Val: {len(val_pairs):,}")

    train_ds = PairDataset(train_pairs, IMG_SIZE, True)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4,
                          pin_memory=True, drop_last=True, persistent_workers=True, prefetch_factor=2)

    optimizer = torch.optim.AdamW(unet.parameters(), lr=LR, betas=(0.9, 0.999), weight_decay=1e-2)
    lr_sched = get_cosine_schedule_with_warmup(optimizer, LR_WARMUP, NUM_TRAIN_STEPS)
    unet, optimizer, train_dl, lr_sched = accelerator.prepare(unet, optimizer, train_dl, lr_sched)

    # ── resume: HF 'bridge_resume' (80k) first; Drive fallback; else fresh ──
    latest = os.path.join(OUTPUT_DIR, "latest")
    start_step = 0; latent_scale = None; resume_dir = None
    if is_main and download_resume_from_hf(latest):
        resume_dir = latest; print("  Resume source: HF (bridge_resume, 80k).")
    accelerator.wait_for_everyone()
    if not resume_dir and os.path.exists(os.path.join(latest, "meta.pt")):
        resume_dir = latest
    if not resume_dir and os.path.exists(os.path.join(DRIVE_LATEST, "meta.pt")):
        resume_dir = DRIVE_LATEST; print("  Resume source: DRIVE (fallback).")
    if resume_dir and os.path.exists(os.path.join(resume_dir, "meta.pt")):
        meta = torch.load(os.path.join(resume_dir, "meta.pt"), map_location="cpu")
        start_step = meta["step"]; latent_scale = meta["latent_scale"]
        accelerator.load_state(resume_dir)
        print(f"  RESUMED @ step {start_step} (latent_scale={latent_scale:.4f}).")
    else:
        if is_main:
            stds = []
            for b in DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2):
                with torch.autocast("cuda", enabled=False):
                    stds.append(vae.encode_latent(b["opt"].to(device).float()).std().item())
                if len(stds) >= 10: break
            latent_scale = 1.0 / float(np.mean(stds))
            print(f"  Computed latent_scale = {latent_scale:.4f}")
        ls = torch.tensor(latent_scale if latent_scale else 1.0, device=device)
        latent_scale = float(accelerator.gather(ls).mean().item()) if accelerator.num_processes > 1 else float(ls)

    if is_main:
        with torch.no_grad(), torch.autocast("cuda", enabled=False):
            vb = next(iter(DataLoader(PairDataset(val_pairs, IMG_SIZE, False), batch_size=8)))
            x = vb["opt"].to(device).float()
            rec = vae.decode_latent(vae.encode_latent(x)).clamp(-1, 1)
            psnr = 10 * np.log10(4.0 / F.mse_loss(rec, x).item())
        print(f"  SANITY: VAE recon PSNR = {psnr:.2f} dB ({'OK' if psnr > 22 else 'LOW!'})")

    def encode_scaled(imgs):
        with torch.no_grad(), torch.autocast("cuda", enabled=False):
            return vae.encode_latent(imgs.float()) * latent_scale

    @torch.no_grad()
    def validate(step):
        sub = random.Random(7).sample(val_pairs, min(N_VAL, len(val_pairs)))
        vdl = DataLoader(PairDataset(sub, IMG_SIZE, False), batch_size=8, num_workers=2)
        P, S, L = [], [], []
        m = accelerator.unwrap_model(unet); m.eval()
        for b in vdl:
            opt = b["opt"].to(device); sar = b["sar"].to(device)
            y = encode_scaled(sar)
            cond = get_cond(b["season"], b["terrain"], embed_cache).to(device)
            x0 = bridge_sample(m, y, cond, eta, KAPPA, dtype)
            with torch.autocast("cuda", enabled=False):
                rec = vae.decode_latent((x0 / latent_scale).float()).clamp(-1, 1)
            P.append(10 * np.log10(4.0 / F.mse_loss(rec, opt).item()))
            S.append(ssim_fn(rec, opt).item()); L.append(lpips_fn(rec, opt).mean().item())
        m.train()
        print(f"\n[VAL step {step}] PSNR {np.mean(P):.3f} | SSIM {np.mean(S):.4f} | LPIPS {np.mean(L):.4f}\n")

    def local_save(step):
        if LOCAL_SAVE_EVERY == 0: return
        accelerator.wait_for_everyone()
        if not is_main: return
        os.makedirs(latest, exist_ok=True)
        accelerator.save_state(latest)                      # LOCAL disk, fast, no upload
        torch.save({"step": step, "latent_scale": latent_scale}, os.path.join(latest, "meta.pt"))
        print(f"  local safety checkpoint @ step {step}")

    def final_save(step):
        accelerator.wait_for_everyone()
        if not is_main: return
        tmp = os.path.join(OUTPUT_DIR, "_final"); os.makedirs(tmp, exist_ok=True)
        sd = {k: v.contiguous().cpu() for k, v in accelerator.unwrap_model(unet).state_dict().items()}
        save_file(sd, os.path.join(tmp, "unet.safetensors"))
        with open(os.path.join(tmp, "bridge_config.json"), "w") as f:
            json.dump({"latent_scale": latent_scale, "T": T_STEPS, "kappa": KAPPA,
                       "eta_1": ETA_1, "eta_T": ETA_T, "img_size": IMG_SIZE,
                       "final_step": step}, f, indent=2)
        push_hf(tmp, "bridge_final")
        shutil.rmtree(tmp, ignore_errors=True)
        print(f"  FINAL model (safetensors) -> HF/bridge_final @ step {step}")

    global_step = start_step
    ema = 0.0; a = 0.98
    spe = len(train_dl); epochs = (NUM_TRAIN_STEPS + spe - 1) // spe
    bar = tqdm(total=NUM_TRAIN_STEPS, initial=start_step, desc="Stage 3 bridge",
               disable=not is_main, dynamic_ncols=True)
    unet.train()
    print(f"\n>>> Resuming @ {start_step}. No periodic HF push. Final safetensors save at the end.\n")

    for epoch in range(epochs):
        for batch in train_dl:
            x0 = encode_scaled(batch["opt"].to(device))
            y  = encode_scaled(batch["sar"].to(device))
            cond = get_cond(batch["season"], batch["terrain"], embed_cache, COND_DROPOUT).to(device)
            t = torch.randint(0, T_STEPS, (x0.shape[0],), device=device)
            x_t = bridge_forward(x0, y, t, eta, KAPPA)
            tb = (t * 1000 // T_STEPS).long()
            with accelerator.accumulate(unet):
                pred = unet(torch.cat([x_t, y], 1).to(dtype), tb,
                            encoder_hidden_states=cond.to(dtype)).sample
                loss = F.mse_loss(pred.float(), x0.float())
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(unet.parameters(), 1.0)
                optimizer.step(); lr_sched.step(); optimizer.zero_grad(set_to_none=True)
            if accelerator.sync_gradients:
                global_step += 1
                ema = a * ema + (1 - a) * loss.item()
                if is_main:
                    bar.update(1)
                    if global_step % LOG_EVERY == 0: bar.set_postfix({"mse": f"{ema:.4f}"})
                if global_step % VAL_EVERY == 0 and is_main: validate(global_step)
                if LOCAL_SAVE_EVERY and global_step % LOCAL_SAVE_EVERY == 0: local_save(global_step)
            if global_step >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if global_step >= NUM_TRAIN_STEPS: break

    bar.close()
    if is_main: validate(global_step)
    final_save(global_step)
    accelerator.end_training()
    print("\nStage 3 complete. Final model on HF/bridge_final (safetensors).")


if __name__ == "__main__":
    main()

Overwriting /content/stage3_bridge_256.py


In [ ]:
import os
os.environ["HF_WRITE_TOKEN"] = "REDACTED_HF_TOKEN"
!pip install -q lpips torchmetrics scipy
!python /content/stage3_bridge_256.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading components...
  Adapted VAE loaded from adapted16_final.
Loading weights: 100% 196/196 [00:00<00:00, 1262.32it/s, Materializing param=text_model.final_layer_norm.weight]
CLIPTextM

In [ ]:
%%writefile /content/stage4_evaluate.py
"""
SAR-INTEL Stage 4 — Evaluation. Loads bridge_final (safetensors) + adapted VAE,
runs the full val set and reports the paper's results:
  - PSNR / SSIM / LPIPS overall, per-season, per-terrain
  - FID (headline metric for the paper) via clean-fid / pytorch-fid fallback
  - steps-ablation (1 / 5 / 15 steps): does iterative refinement help or collapse?
  - resolution: 256px (benchmark-comparable) AND 512px
  - visual comparison grids (SAR | predicted | ground-truth) per season
Inference only. No training. ~1-2 hours.
"""
import os, gc, re, json, random, warnings, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
from huggingface_hub import hf_hub_download, HfApi
from safetensors.torch import load_file
import lpips as lpips_lib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance
from sklearn.model_selection import train_test_split
from collections import defaultdict

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
DATA_ROOT   = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR  = "/content/stage4_eval"
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID  = "AliMusaRizvi/sar-to-optical-diffusion"
HF_TOKEN    = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG     = "adapted16_final"
BRIDGE_TAG  = "bridge_final"

SEASONS     = ("spring", "summer", "fall", "winter")
TERRAINS    = ("tropical", "temperate", "arctic", "arid", "coastal", "urban")
VAL_FRACTION= 0.05
SEED        = 42
BASE_CH, DETAIL_CH, TOTAL_CH = 4, 12, 16

# evaluation scope
EVAL_RES        = (256, 512)        # headline 256, also report 512
N_EVAL          = 1000             # val patches for metrics (set None for full val set)
N_FID           = 1000             # patches for FID (more = more stable)
STEPS_ABLATION  = (1, 5, 15)       # check refinement vs single-step collapse
N_GRID_PER_SEASON = 3
BATCH           = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEASON_DESC = {"spring": "spring", "summer": "summer", "fall": "autumn", "winter": "winter"}

def map_region_to_terrain(region):
    r = (region or "").lower()
    if any(k in r for k in ("trop", "rainforest", "amazon", "jungle")): return "tropical"
    if any(k in r for k in ("arctic", "tundra", "snow", "ice", "polar")): return "arctic"
    if any(k in r for k in ("desert", "arid", "sahara", "dry")): return "arid"
    if any(k in r for k in ("coast", "beach", "shore", "marine", "sea")): return "coastal"
    if any(k in r for k in ("urban", "city", "metro")): return "urban"
    return "temperate"


def collect_pairs(root, seasons):
    root_str = Path(root).as_posix(); pairs = []
    for season in seasons:
        csvs = list((Path(root) / season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for col in ("s1_fileName", "s2_fileName"):
            df[col] = df[col].astype(str).str.replace("\\", "/", regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            pairs.append({"sar": f"{root_str}/{row['s1_fileName']}",
                          "opt": f"{root_str}/{row['s2_fileName']}",
                          "season": season, "terrain": map_region_to_terrain(row["region"])})
    return pairs


class PairDataset(Dataset):
    def __init__(self, pairs, img_size):
        self.pairs = pairs; self.img_size = img_size
    def __len__(self): return len(self.pairs)
    def _load(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1) * 2.0 - 1.0
    def __getitem__(self, idx):
        p = self.pairs[idx]
        return {"sar": self._load(p["sar"]), "opt": self._load(p["opt"]),
                "season": p["season"], "terrain": p["terrain"]}


# ── Model defs (must match Stage 3) ───────────────────────────────────────────
class DetailEncoder(nn.Module):
    def __init__(self, in_ch=3, out_ch=DETAIL_CH, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, nf, 3, 2, 1),  nn.GroupNorm(8, nf), nn.SiLU(),
            nn.Conv2d(nf, nf*2, 3, 2, 1),   nn.GroupNorm(8, nf*2), nn.SiLU(),
            nn.Conv2d(nf*2, nf*4, 3, 2, 1), nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, nf*4, 3, 1, 1), nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, out_ch, 3, 1, 1))
    def forward(self, x): return self.net(x)


class AdaptedVAE(nn.Module):
    def __init__(self, base_vae):
        super().__init__()
        self.base_vae = base_vae; self.detail_enc = DetailEncoder()
        old = base_vae.decoder.conv_in
        base_vae.decoder.conv_in = nn.Conv2d(TOTAL_CH, old.out_channels, old.kernel_size,
                                             old.stride, old.padding, bias=(old.bias is not None))
    @torch.no_grad()
    def encode_latent(self, x):
        z = self.base_vae.encode(x).latent_dist.mode()
        return torch.cat([self.base_vae.post_quant_conv(z), self.detail_enc(x)], dim=1)
    @torch.no_grad()
    def decode_latent(self, lat):
        return self.base_vae.decoder(lat)


def load_models():
    base_vae = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    vae = AdaptedVAE(base_vae)
    bp = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/base_vae.pth")
    de = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/detail_encoder.pth")
    vae.base_vae.load_state_dict(torch.load(bp, map_location="cpu"))
    vae.detail_enc.load_state_dict(torch.load(de, map_location="cpu"))
    vae = vae.to(device).eval().float(); vae.requires_grad_(False)

    cfg_path = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{BRIDGE_TAG}/bridge_config.json")
    cfg = json.load(open(cfg_path))
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    unet.conv_in = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    oo = unet.conv_out
    unet.conv_out = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    unet.config.in_channels = 2*TOTAL_CH; unet.config.out_channels = TOTAL_CH
    sd_path = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{BRIDGE_TAG}/unet.safetensors")
    unet.load_state_dict(load_file(sd_path))
    unet = unet.to(device).eval(); unet.requires_grad_(False)
    print(f"  Loaded bridge_final (step {cfg.get('final_step')}), latent_scale={cfg['latent_scale']:.4f}")
    return vae, unet, cfg


def build_embed_cache():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    txt.requires_grad_(False); cache = {}
    @torch.no_grad()
    def embed(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s, ter)] = embed(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = embed(""); del txt; gc.collect(); torch.cuda.empty_cache()
    return cache


def make_eta(T, e1, eT):
    idx = np.arange(T)
    return torch.tensor(e1 * (eT / e1) ** (idx / (T - 1)), dtype=torch.float32)


def get_cond(seasons, terrains, cache):
    return torch.stack([cache.get((s, t), cache["null"]) for s, t in zip(seasons, terrains)])


@torch.no_grad()
def sample(unet, y, cond, eta_full, kappa, n_steps):
    """ResShift sampling with n_steps (subsampled from the full T schedule)."""
    T = len(eta_full)
    idx = np.linspace(0, T - 1, n_steps).round().astype(int)
    eta = eta_full[idx]
    B = y.shape[0]
    x = y + kappa * torch.sqrt(eta[-1]).to(y.device) * torch.randn_like(y)
    for k in reversed(range(len(eta))):
        tb = torch.full((B,), int(idx[k] * 1000 / T), device=y.device, dtype=torch.long)
        x0 = unet(torch.cat([x, y], 1).to(torch.bfloat16), tb,
                  encoder_hidden_states=cond.to(torch.bfloat16)).sample.float()
        if k > 0:
            ep = eta[k - 1].to(y.device)
            x = x0 + ep * (y - x0) + kappa * torch.sqrt(ep) * torch.randn_like(x)
        else:
            x = x0
    return x


def denorm01(t): return (t.clamp(-1, 1) + 1) / 2


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    vae, unet, cfg = load_models()
    embed_cache = build_embed_cache()
    scale = cfg["latent_scale"]; kappa = cfg["kappa"]
    eta_full = make_eta(cfg["T"], cfg["eta_1"], cfg["eta_T"]).to(device)

    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)

    all_pairs = collect_pairs(Path(DATA_ROOT), SEASONS)
    _, val_pairs = train_test_split(
        all_pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in all_pairs], random_state=SEED)
    eval_pairs = val_pairs if N_EVAL is None else random.Random(SEED).sample(val_pairs, min(N_EVAL, len(val_pairs)))
    print(f"Evaluating on {len(eval_pairs):,} val patches.")

    def enc(imgs):
        with torch.no_grad(), torch.autocast("cuda", enabled=False):
            return vae.encode_latent(imgs.float()) * scale

    @torch.no_grad()
    def run_eval(res, n_steps, want_fid=False, want_breakdown=False):
        ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)
        fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device) if want_fid else None
        dl = DataLoader(PairDataset(eval_pairs, res), batch_size=BATCH, num_workers=4)
        agg = defaultdict(list)
        by_season = defaultdict(lambda: defaultdict(list))
        by_terrain = defaultdict(lambda: defaultdict(list))
        for b in dl:
            opt = b["opt"].to(device); sar = b["sar"].to(device)
            y = enc(sar)
            cond = get_cond(b["season"], b["terrain"], embed_cache).to(device)
            x0 = sample(unet, y, cond, eta_full, kappa, n_steps)
            with torch.autocast("cuda", enabled=False):
                rec = vae.decode_latent((x0 / scale).float()).clamp(-1, 1)
            for i in range(rec.shape[0]):
                p = 10 * np.log10(4.0 / F.mse_loss(rec[i:i+1], opt[i:i+1]).item())
                s = ssim_fn(rec[i:i+1], opt[i:i+1]).item()
                l = lpips_fn(rec[i:i+1], opt[i:i+1]).item()
                agg["psnr"].append(p); agg["ssim"].append(s); agg["lpips"].append(l)
                if want_breakdown:
                    se, te = b["season"][i], b["terrain"][i]
                    for k, v in (("psnr", p), ("ssim", s), ("lpips", l)):
                        by_season[se][k].append(v); by_terrain[te][k].append(v)
            if want_fid:
                fid.update(denorm01(opt), real=True)
                fid.update(denorm01(rec), real=False)
        out = {k: float(np.mean(v)) for k, v in agg.items()}
        if want_fid: out["fid"] = float(fid.compute().item())
        if want_breakdown:
            out["by_season"] = {s: {k: float(np.mean(v)) for k, v in d.items()} for s, d in by_season.items()}
            out["by_terrain"] = {t: {k: float(np.mean(v)) for k, v in d.items()} for t, d in by_terrain.items()}
        return out

    results = {}

    # 1. Headline @256px, full T steps, with FID + breakdown
    print("\n=== 256px (headline) ===")
    r256 = run_eval(256, cfg["T"], want_fid=True, want_breakdown=True)
    results["256px"] = r256
    print(f"  PSNR {r256['psnr']:.3f} | SSIM {r256['ssim']:.4f} | LPIPS {r256['lpips']:.4f} | FID {r256['fid']:.3f}")
    print("  Per-season:")
    for s, d in r256["by_season"].items():
        print(f"    {s:8s} PSNR {d['psnr']:.2f} | SSIM {d['ssim']:.3f} | LPIPS {d['lpips']:.3f}")
    print("  Per-terrain:")
    for t, d in r256["by_terrain"].items():
        print(f"    {t:10s} PSNR {d['psnr']:.2f} | SSIM {d['ssim']:.3f} | LPIPS {d['lpips']:.3f}")

    # 2. 512px (secondary, comparability)
    print("\n=== 512px (secondary) ===")
    r512 = run_eval(512, cfg["T"], want_fid=True)
    results["512px"] = r512
    print(f"  PSNR {r512['psnr']:.3f} | SSIM {r512['ssim']:.4f} | LPIPS {r512['lpips']:.4f} | FID {r512['fid']:.3f}")

    # 3. Steps-ablation @256px (refinement vs single-step collapse)
    print("\n=== steps-ablation @256px ===")
    abl = {}
    for ns in STEPS_ABLATION:
        r = run_eval(256, ns, want_fid=False)
        abl[ns] = r
        print(f"  {ns:2d} steps: PSNR {r['psnr']:.3f} | SSIM {r['ssim']:.4f} | LPIPS {r['lpips']:.4f}")
    results["steps_ablation"] = abl
    spread = max(abl[ns]["psnr"] for ns in abl) - min(abl[ns]["psnr"] for ns in abl)
    print(f"  PSNR spread across steps: {spread:.3f} dB "
          f"({'refinement helps' if spread > 0.3 else 'collapsed to ~single-step'})")

    # 4. Visual grids per season @256px
    print("\n=== visual grids ===")
    grid_pairs = []
    for s in SEASONS:
        pool = [p for p in eval_pairs if p["season"] == s]
        grid_pairs += random.Random(1).sample(pool, min(N_GRID_PER_SEASON, len(pool)))
    n = len(grid_pairs)
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    if n == 1: axes = axes.reshape(1, -1)
    with torch.no_grad():
        for i, p in enumerate(grid_pairs):
            ds = PairDataset([p], 256)[0]
            sar = ds["sar"].unsqueeze(0).to(device); opt = ds["opt"].unsqueeze(0).to(device)
            y = enc(sar); cond = get_cond([p["season"]], [p["terrain"]], embed_cache).to(device)
            x0 = sample(unet, y, cond, eta_full, kappa, cfg["T"])
            with torch.autocast("cuda", enabled=False):
                rec = vae.decode_latent((x0 / scale).float()).clamp(-1, 1)
            ps = 10 * np.log10(4.0 / F.mse_loss(rec, opt).item())
            for j, (img, ttl) in enumerate([(sar[0], f"SAR ({p['season']})"),
                                            (rec[0], f"Predicted {ps:.1f}dB"),
                                            (opt[0], "Ground truth")]):
                axes[i, j].imshow(denorm01(img).permute(1, 2, 0).cpu().numpy())
                axes[i, j].set_title(ttl, fontsize=8); axes[i, j].axis("off")
    plt.tight_layout()
    gpath = os.path.join(OUTPUT_DIR, "eval_grid.png")
    plt.savefig(gpath, dpi=150, bbox_inches="tight"); display(plt.gcf()); plt.close()

    # save + push results
    with open(os.path.join(OUTPUT_DIR, "results.json"), "w") as f:
        json.dump(results, f, indent=2)
    print("\n=== SUMMARY (paper headline) ===")
    print(f"  256px: PSNR {r256['psnr']:.2f} | SSIM {r256['ssim']:.3f} | "
          f"LPIPS {r256['lpips']:.3f} | FID {r256['fid']:.2f}")
    if HF_TOKEN:
        try:
            HfApi().upload_folder(folder_path=OUTPUT_DIR, path_in_repo="eval_results",
                                  repo_id=HF_REPO_ID, repo_type="model", token=HF_TOKEN)
            print("  results -> HF/eval_results")
        except Exception as e:
            print(f"  push failed: {e}")


if __name__ == "__main__":
    main()

Writing /content/stage4_evaluate.py


In [ ]:
import os
os.environ["HF_WRITE_TOKEN"] = "REDACTED_HF_TOKEN"
!pip install -q lpips torchmetrics safetensors scipy
!python /content/stage4_evaluate.py

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
bridge_config.json: 100% 144/144 [00:00<00:00, 577kB/s]
bridge_final/unet.safetensors: 100% 3.44G/3.44G [00:35<00:00, 96.1MB/s]
  Loaded bridge_final (step 150000), latent_scale=1.4010
Lo

In [ ]:
%%writefile /content/stage3b_flow_256.py
"""
SAR-INTEL Stage 3b — FLOW-MATCHING bridge on the adapted 16ch latent @ 256px.
Modern rectified-flow formulation to break the ResShift mean-collapse plateau.

Key upgrades vs ResShift:
  - VELOCITY prediction (rectified flow): x_t=(1-t)x0+t*x1, target v=x1-x0.
  - DECODED PIXEL LOSS (anti-collapse): decode predicted x1, L1+LPIPS vs real
    optical image. This is what fights the blur/mean-collapse that capped PSNR.
  - Euler-ODE sampling from SAR latent -> optical latent.

Saving: NO periodic HF push. LOCAL safety checkpoint overwrites a single folder.
Final model saved ONCE to HF in safetensors. Resumable from HF 'flow_resume'.
"""
import os, gc, re, json, random, warnings, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from accelerate.utils import set_seed
from tqdm.auto import tqdm
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from safetensors.torch import save_file
import lpips as lpips_lib
import pandas as pd
from torchmetrics.image import StructuralSimilarityIndexMeasure
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ── Config ────────────────────────────────────────────────────────────────────
DATA_ROOT    = "/root/.cache/kagglehub/datasets/shambac/augmented-sentinel-1-2/versions/1"
OUTPUT_DIR   = "/content/stage3b_flow"
SD_MODEL_ID  = "runwayml/stable-diffusion-v1-5"
HF_REPO_ID   = "AliMusaRizvi/sar-to-optical-diffusion"
HF_TOKEN     = os.environ.get("HF_WRITE_TOKEN", "")
VAE_TAG      = "adapted16_final"
RESUME_TAG   = "flow_resume"

IMG_SIZE     = 256
SEASONS      = ("spring", "summer", "fall", "winter")
VAL_FRACTION = 0.05
SEED         = 42
BASE_CH, DETAIL_CH, TOTAL_CH = 4, 12, 16

NUM_TRAIN_STEPS = 120_000
BATCH_SIZE   = 4          # smaller: decoded-pixel loss holds a grad graph through the decoder
LR           = 6e-5
LR_WARMUP    = 1_000
COND_DROPOUT = 0.10
PIXEL_START  = 2_000      # warm up velocity loss before adding pixel loss
LAMBDA_LPIPS = 1.0        # decoded LPIPS (anti-collapse) — the key term
LAMBDA_L1    = 0.5        # decoded L1
SAMPLE_STEPS = 10         # Euler steps for validation
VAL_EVERY    = 5_000
LOCAL_SAVE_EVERY = 30_000
N_VAL        = 100
LOG_EVERY    = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEASON_DESC = {"spring": "spring", "summer": "summer", "fall": "autumn", "winter": "winter"}
TERRAINS = ("tropical", "temperate", "arctic", "arid", "coastal", "urban")

def map_region_to_terrain(region):
    r = (region or "").lower()
    if any(k in r for k in ("trop", "rainforest", "amazon", "jungle")): return "tropical"
    if any(k in r for k in ("arctic", "tundra", "snow", "ice", "polar")): return "arctic"
    if any(k in r for k in ("desert", "arid", "sahara", "dry")): return "arid"
    if any(k in r for k in ("coast", "beach", "shore", "marine", "sea")): return "coastal"
    if any(k in r for k in ("urban", "city", "metro")): return "urban"
    return "temperate"


def collect_pairs(root, seasons):
    root_str = Path(root).as_posix(); pairs = []
    for season in seasons:
        csvs = list((Path(root) / season).glob("*.csv"))
        if not csvs: continue
        df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        df.columns = [c.strip() for c in df.columns]
        df["region"] = df["region"].astype(str).str.strip().str.lower()
        for col in ("s1_fileName", "s2_fileName"):
            df[col] = df[col].astype(str).str.replace("\\", "/", regex=False)
        df["s1_fileName"] = df["s1_fileName"].str.replace(r"(\w+?)1s2_", r"\1_s1_", regex=True)
        for _, row in df.iterrows():
            pairs.append({"sar": f"{root_str}/{row['s1_fileName']}",
                          "opt": f"{root_str}/{row['s2_fileName']}",
                          "season": season, "terrain": map_region_to_terrain(row["region"])})
    random.shuffle(pairs); return pairs


class PairDataset(Dataset):
    def __init__(self, pairs, img_size=256, augment=True):
        self.pairs = pairs; self.img_size = img_size; self.augment = augment
    def __len__(self): return len(self.pairs)
    def _load(self, path):
        img = Image.open(path).convert("RGB")
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1) * 2.0 - 1.0
    def __getitem__(self, idx):
        p = self.pairs[idx]
        sar = self._load(p["sar"]); opt = self._load(p["opt"])
        if self.augment:
            if random.random() > 0.5: sar, opt = torch.flip(sar, [2]), torch.flip(opt, [2])
            if random.random() > 0.5: sar, opt = torch.flip(sar, [1]), torch.flip(opt, [1])
        return {"sar": sar, "opt": opt, "season": p["season"], "terrain": p["terrain"]}


class DetailEncoder(nn.Module):
    def __init__(self, in_ch=3, out_ch=DETAIL_CH, nf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, nf, 3, 2, 1),  nn.GroupNorm(8, nf), nn.SiLU(),
            nn.Conv2d(nf, nf*2, 3, 2, 1),   nn.GroupNorm(8, nf*2), nn.SiLU(),
            nn.Conv2d(nf*2, nf*4, 3, 2, 1), nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, nf*4, 3, 1, 1), nn.GroupNorm(8, nf*4), nn.SiLU(),
            nn.Conv2d(nf*4, out_ch, 3, 1, 1))
    def forward(self, x): return self.net(x)


class AdaptedVAE(nn.Module):
    def __init__(self, base_vae):
        super().__init__()
        self.base_vae = base_vae; self.detail_enc = DetailEncoder()
        old = base_vae.decoder.conv_in
        base_vae.decoder.conv_in = nn.Conv2d(TOTAL_CH, old.out_channels, old.kernel_size,
                                             old.stride, old.padding, bias=(old.bias is not None))
    @torch.no_grad()
    def encode_latent(self, x):
        z = self.base_vae.encode(x).latent_dist.mode()
        return torch.cat([self.base_vae.post_quant_conv(z), self.detail_enc(x)], dim=1)
    def decode_grad(self, lat):                 # grad-enabled (params frozen, grad flows to input)
        return self.base_vae.decoder(lat)
    @torch.no_grad()
    def decode_latent(self, lat):
        return self.base_vae.decoder(lat)


def load_adapted_vae():
    base_vae = AutoencoderKL.from_pretrained(SD_MODEL_ID, subfolder="vae")
    vae = AdaptedVAE(base_vae)
    bp = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/base_vae.pth")
    de = hf_hub_download(repo_id=HF_REPO_ID, filename=f"{VAE_TAG}/detail_encoder.pth")
    vae.base_vae.load_state_dict(torch.load(bp, map_location="cpu"))
    vae.detail_enc.load_state_dict(torch.load(de, map_location="cpu"))
    vae = vae.to(device).eval().float(); vae.requires_grad_(False)
    print(f"  Adapted VAE loaded from {VAE_TAG}.")
    return vae


def build_embed_cache():
    tok = CLIPTokenizer.from_pretrained(SD_MODEL_ID, subfolder="tokenizer")
    txt = CLIPTextModel.from_pretrained(SD_MODEL_ID, subfolder="text_encoder").to(device).eval()
    txt.requires_grad_(False); cache = {}
    @torch.no_grad()
    def embed(p):
        t = tok(p, padding="max_length", max_length=tok.model_max_length,
                truncation=True, return_tensors="pt").input_ids.to(device)
        return txt(t)[0][0].cpu()
    for s in SEASONS:
        for ter in TERRAINS:
            cache[(s, ter)] = embed(f"a {SEASON_DESC[s]} satellite optical image of {ter} terrain")
    cache["null"] = embed(""); del txt; gc.collect(); torch.cuda.empty_cache()
    print(f"  Built {len(cache)} CLIP embeddings.")
    return cache


def build_unet():
    unet = UNet2DConditionModel.from_pretrained(SD_MODEL_ID, subfolder="unet")
    oi = unet.conv_in
    ni = nn.Conv2d(2*TOTAL_CH, oi.out_channels, oi.kernel_size, oi.stride, oi.padding)
    with torch.no_grad():
        ni.weight.zero_()
        for i in range(2*TOTAL_CH): ni.weight[:, i] = oi.weight[:, i % oi.in_channels] / 8.0
        ni.bias.copy_(oi.bias)
    unet.conv_in = ni; unet.config.in_channels = 2*TOTAL_CH
    oo = unet.conv_out
    no = nn.Conv2d(oo.in_channels, TOTAL_CH, oo.kernel_size, oo.stride, oo.padding)
    with torch.no_grad():
        for i in range(TOTAL_CH):
            no.weight[i] = oo.weight[i % oo.out_channels]; no.bias[i] = oo.bias[i % oo.out_channels]
    unet.conv_out = no; unet.config.out_channels = TOTAL_CH
    print(f"  UNet: conv_in {2*TOTAL_CH}ch, conv_out {TOTAL_CH}ch (flow-matching).")
    return unet


def get_cond(seasons, terrains, cache, drop=0.0):
    out = []
    for s, ter in zip(seasons, terrains):
        out.append(cache["null"] if (drop > 0 and random.random() < drop)
                   else cache.get((s, ter), cache["null"]))
    return torch.stack(out)


@torch.no_grad()
def flow_sample(unet, x0, cond, n_steps, dtype):
    """Euler ODE from SAR latent x0 (t=0) to optical latent (t=1)."""
    x = x0.clone(); dt = 1.0 / n_steps; B = x0.shape[0]
    for i in range(n_steps):
        t = i * dt
        tb = torch.full((B,), int(t * 999), device=x0.device, dtype=torch.long)
        v = unet(torch.cat([x, x0], 1).to(dtype), tb, encoder_hidden_states=cond.to(dtype)).sample.float()
        x = x + dt * v
    return x


def push_hf(folder, tag):
    if not HF_TOKEN: return
    try:
        HfApi().upload_folder(folder_path=folder, path_in_repo=tag, repo_id=HF_REPO_ID,
                              repo_type="model", token=HF_TOKEN)
        print(f"  HF push -> {tag}")
    except Exception as e:
        print(f"  HF push failed: {e}")


def download_resume_from_hf(dest):
    try:
        snapshot_download(repo_id=HF_REPO_ID, allow_patterns=f"{RESUME_TAG}/*",
                          local_dir="/content/_flow_resume_dl", token=HF_TOKEN)
        src = os.path.join("/content/_flow_resume_dl", RESUME_TAG)
        if os.path.exists(os.path.join(src, "meta.pt")):
            if os.path.exists(dest): shutil.rmtree(dest)
            shutil.copytree(src, dest); return True
    except Exception as e:
        print(f"  no HF resume state ({e})")
    return False


def main():
    accelerator = Accelerator(mixed_precision="bf16", project_dir=OUTPUT_DIR)
    set_seed(SEED); is_main = accelerator.is_main_process; dtype = torch.bfloat16
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("Loading components...")
    vae = load_adapted_vae(); embed_cache = build_embed_cache(); unet = build_unet()
    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)
    ssim_fn = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)

    all_pairs = collect_pairs(Path(DATA_ROOT), SEASONS)
    train_pairs, val_pairs = train_test_split(
        all_pairs, test_size=VAL_FRACTION,
        stratify=[f"{p['season']}_{p['terrain']}" for p in all_pairs], random_state=SEED)
    print(f"Train: {len(train_pairs):,} | Val: {len(val_pairs):,}")

    train_ds = PairDataset(train_pairs, IMG_SIZE, True)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4,
                          pin_memory=True, drop_last=True, persistent_workers=True, prefetch_factor=2)

    optimizer = torch.optim.AdamW(unet.parameters(), lr=LR, betas=(0.9, 0.999), weight_decay=1e-2)
    lr_sched = get_cosine_schedule_with_warmup(optimizer, LR_WARMUP, NUM_TRAIN_STEPS)
    unet, optimizer, train_dl, lr_sched = accelerator.prepare(unet, optimizer, train_dl, lr_sched)

    latest = os.path.join(OUTPUT_DIR, "latest")
    start_step = 0; latent_scale = None; resume_dir = None
    if is_main and download_resume_from_hf(latest):
        resume_dir = latest; print("  Resume source: HF (flow_resume).")
    accelerator.wait_for_everyone()
    if not resume_dir and os.path.exists(os.path.join(latest, "meta.pt")):
        resume_dir = latest
    if resume_dir and os.path.exists(os.path.join(resume_dir, "meta.pt")):
        meta = torch.load(os.path.join(resume_dir, "meta.pt"), map_location="cpu")
        start_step = meta["step"]; latent_scale = meta["latent_scale"]
        accelerator.load_state(resume_dir)
        print(f"  RESUMED @ step {start_step} (latent_scale={latent_scale:.4f}).")
    else:
        if is_main:
            stds = []
            for b in DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2):
                with torch.autocast("cuda", enabled=False):
                    stds.append(vae.encode_latent(b["opt"].to(device).float()).std().item())
                if len(stds) >= 10: break
            latent_scale = 1.0 / float(np.mean(stds))
            print(f"  Computed latent_scale = {latent_scale:.4f}")
        ls = torch.tensor(latent_scale if latent_scale else 1.0, device=device)
        latent_scale = float(accelerator.gather(ls).mean().item()) if accelerator.num_processes > 1 else float(ls)

    def enc(imgs):
        with torch.no_grad(), torch.autocast("cuda", enabled=False):
            return vae.encode_latent(imgs.float()) * latent_scale

    @torch.no_grad()
    def validate(step):
        sub = random.Random(7).sample(val_pairs, min(N_VAL, len(val_pairs)))
        vdl = DataLoader(PairDataset(sub, IMG_SIZE, False), batch_size=8, num_workers=2)
        P, S, L = [], [], []
        m = accelerator.unwrap_model(unet); m.eval()
        for b in vdl:
            opt = b["opt"].to(device)
            x0 = enc(b["sar"].to(device))
            cond = get_cond(b["season"], b["terrain"], embed_cache).to(device)
            x1 = flow_sample(m, x0, cond, SAMPLE_STEPS, dtype)
            with torch.autocast("cuda", enabled=False):
                rec = vae.decode_latent((x1 / latent_scale).float()).clamp(-1, 1)
            P.append(10 * np.log10(4.0 / F.mse_loss(rec, opt).item()))
            S.append(ssim_fn(rec, opt).item()); L.append(lpips_fn(rec, opt).mean().item())
        m.train()
        print(f"\n[VAL step {step}] PSNR {np.mean(P):.3f} | SSIM {np.mean(S):.4f} | LPIPS {np.mean(L):.4f}\n")

    def local_save(step):
        if LOCAL_SAVE_EVERY == 0: return
        accelerator.wait_for_everyone()
        if not is_main: return
        if os.path.exists(latest): shutil.rmtree(latest)        # overwrite, never grows
        os.makedirs(latest, exist_ok=True)
        accelerator.save_state(latest)
        torch.save({"step": step, "latent_scale": latent_scale}, os.path.join(latest, "meta.pt"))
        print(f"  local safety checkpoint (overwritten) @ step {step}")

    def final_save(step):
        accelerator.wait_for_everyone()
        if not is_main: return
        tmp = os.path.join(OUTPUT_DIR, "_final"); os.makedirs(tmp, exist_ok=True)
        sd = {k: v.contiguous().cpu() for k, v in accelerator.unwrap_model(unet).state_dict().items()}
        save_file(sd, os.path.join(tmp, "unet.safetensors"))
        with open(os.path.join(tmp, "bridge_config.json"), "w") as f:
            json.dump({"latent_scale": latent_scale, "method": "flow_matching",
                       "sample_steps": SAMPLE_STEPS, "img_size": IMG_SIZE, "final_step": step}, f, indent=2)
        push_hf(tmp, "flow_final")
        shutil.rmtree(tmp, ignore_errors=True)
        print(f"  FINAL flow model -> HF/flow_final @ step {step}")

    global_step = start_step
    ema_v = ema_p = 0.0; a = 0.98
    spe = len(train_dl); epochs = (NUM_TRAIN_STEPS + spe - 1) // spe
    bar = tqdm(total=NUM_TRAIN_STEPS, initial=start_step, desc="Stage 3b flow",
               disable=not is_main, dynamic_ncols=True)
    unet.train()
    print(f"\n>>> Flow-matching from step {start_step}. Pixel loss kicks in at {PIXEL_START}.\n")

    for epoch in range(epochs):
        for batch in train_dl:
            opt_img = batch["opt"].to(device)
            x0 = enc(batch["sar"].to(device))        # SAR latent (source)
            x1 = enc(opt_img)                         # optical latent (target)
            cond = get_cond(batch["season"], batch["terrain"], embed_cache, COND_DROPOUT).to(device)
            t = torch.rand(x0.shape[0], device=device)            # continuous U(0,1)
            tv = t.view(-1, 1, 1, 1)
            x_t = (1 - tv) * x0 + tv * x1                          # rectified-flow path
            v_target = x1 - x0
            tb = (t * 999).long()

            with accelerator.accumulate(unet):
                v_pred = unet(torch.cat([x_t, x0], 1).to(dtype), tb,
                              encoder_hidden_states=cond.to(dtype)).sample
                loss_v = F.mse_loss(v_pred.float(), v_target.float())
                loss = loss_v
                loss_p_val = 0.0
                if global_step >= PIXEL_START:
                    x1_pred = x_t + (1 - tv) * v_pred.float()      # reconstruct target latent
                    with torch.autocast("cuda", enabled=False):
                        img_pred = vae.decode_grad((x1_pred / latent_scale).float()).clamp(-1, 1)
                    loss_p = LAMBDA_L1 * F.l1_loss(img_pred, opt_img) + \
                             LAMBDA_LPIPS * lpips_fn(img_pred, opt_img).mean()
                    loss = loss + loss_p
                    loss_p_val = float(loss_p.item())
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(unet.parameters(), 1.0)
                optimizer.step(); lr_sched.step(); optimizer.zero_grad(set_to_none=True)

            if accelerator.sync_gradients:
                global_step += 1
                ema_v = a*ema_v + (1-a)*loss_v.item(); ema_p = a*ema_p + (1-a)*loss_p_val
                if is_main:
                    bar.update(1)
                    if global_step % LOG_EVERY == 0:
                        bar.set_postfix({"v": f"{ema_v:.4f}", "pix": f"{ema_p:.4f}"})
                if global_step % VAL_EVERY == 0 and is_main: validate(global_step)
                if LOCAL_SAVE_EVERY and global_step % LOCAL_SAVE_EVERY == 0: local_save(global_step)
            if global_step >= NUM_TRAIN_STEPS: break
        gc.collect(); torch.cuda.empty_cache()
        if global_step >= NUM_TRAIN_STEPS: break

    bar.close()
    if is_main: validate(global_step)
    final_save(global_step)
    accelerator.end_training()
    print("\nStage 3b complete. Flow model on HF/flow_final.")


if __name__ == "__main__":
    main()

Writing /content/stage3b_flow_256.py


In [ ]:
import os
os.environ["HF_WRITE_TOKEN"] = "REDACTED_HF_TOKEN"
!pip install -q lpips torchmetrics safetensors scipy
!python /content/stage3b_flow_256.py

Streaming output truncated to the last 5000 lines.
Stage 3b flow:  43% 52030/120000 [4:53:55<6:16:25,  3.01it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52031/120000 [4:53:56<6:16:03,  3.01it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52032/120000 [4:53:56<6:17:06,  3.00it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52033/120000 [4:53:56<6:16:42,  3.01it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52034/120000 [4:53:56<6:15:58,  3.01it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52035/120000 [4:53:57<6:15:33,  3.02it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52036/120000 [4:53:57<6:15:38,  3.02it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52037/120000 [4:53:57<6:15:19,  3.02it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52038/120000 [4:53:58<6:15:44,  3.01it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52039/120000 [4:53:58<6:16:49,  3.01it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52040/120000 [4:53:58<6:17:02,  3.00it/s, v=0.2087, pix=0.2514]
Stage 3b flow:  43% 52041